# 01 — Seleção de bases, limpeza de runs e construção das tabelas base × família

Este notebook consolida a Fase 1 do experimento de recomendação de classificadores sobre a suíte **OpenML-CC18**: filtra as bases, valida e limpa os *runs* degenerados, agrupa *flows* em famílias, agrega o desempenho por célula (`max_sub3`) e define o núcleo sem faltantes.

> **Aviso de execução:** As células de consulta à API do OpenML na **Seção 1** (`cobertura_tasks.csv`, testes de paginação e `contagem_runs.csv`) são lentas e não precisam ser reexecutadas; os dados brutos já estão persistidos em `../data/`.

| Seção | Lê | Grava |
| :--- | :--- | :--- |
| **1. Seleção e extração** | API OpenML (`OpenML-CC18`), `data/runs/task_*`, `data/flows.csv` | `data/cobertura_tasks.csv`, `data/contagem_runs.csv` |
| **2. Integridade e limpeza** | `data/runs/task_*`, `data/contagem_runs.csv` | `data/runs_consolidado.parquet`, `data/runs_limpos.parquet`, `data/runs_limpos_v2.parquet`, `data/runs_limpos_v3.parquet` |
| **3. Famílias** | `data/runs_limpos_v3.parquet`, `data/flows.csv` | `data/runs_v3_familias.parquet`, `data/runs_v3_familia.parquet`, `data/familias.csv` |
| **4. Agregação e tabelas** | `data/runs_consolidado.parquet` (diagnóstico) | `data/tabelas/tab_sub3.*`, `tab_max.*`, `tab_med.*`, `tab_n.*`, `celulas_longa.parquet` |
| **5. Faltantes e núcleo** | `data/tabelas/tab_sub3.parquet` | `data/tabelas/nucleo_sub3.parquet`, `nucleo_sub3.csv` |
| **6. Diagnósticos** | `data/tabelas/tab_sub3.parquet`, `tab_n.parquet`, `nucleo_sub3.parquet` | `data/tabelas/tab_sub3_sem_limpeza.parquet`, `tab_sub3_sem_fracas.parquet`, `nucleo_sub3_sem_fracas.parquet` |

*Arquivos legados em `data/` (sensibilidade antiga de 90 famílias com mínimo de 1 run, não utilizados neste pipeline): `celulas_base_familia.csv` e `runs_90familias.csv.gz`.*

## 1.Seleção e extração

In [1]:
import openml
import pandas as pd

suite = openml.study.get_suite("OpenML-CC18")
tasks = openml.tasks.list_tasks(task_id=suite.tasks, output_format="dataframe")
print(len(tasks))
print(tasks.columns.tolist())

72
['tid', 'ttid', 'did', 'name', 'task_type', 'status', 'estimation_procedure', 'source_data', 'target_feature', 'MajorityClassSize', 'MaxNominalAttDistinctValues', 'MinorityClassSize', 'NumberOfClasses', 'NumberOfFeatures', 'NumberOfInstances', 'NumberOfInstancesWithMissingValues', 'NumberOfMissingValues', 'NumberOfNumericFeatures', 'NumberOfSymbolicFeatures']


In [ ]:
print(tasks.estimation_procedure.value_counts())

selected_tasks = tasks[
    tasks.NumberOfInstances.between(500, 50000) &
    (tasks.NumberOfFeatures <= 500)
]

print(len(selected_tasks))

estimation_procedure
10-fold Crossvalidation    72
Name: count, dtype: int64
60


In [3]:
print((tasks.NumberOfInstances > 50_000).sum(), "tasks com mais de 50.000 instâncias")
print((tasks.NumberOfFeatures > 500).sum(), "tasks com mais de 500 features")
print((tasks.NumberOfInstances < 500).sum(), "tasks com menos de 500 instâncias")

6 tasks com mais de 50.000 instâncias
10 tasks com mais de 500 features
0 tasks com menos de 500 instâncias


In [4]:
import os, time
import pandas as pd

arq = "../data/cobertura_tasks.csv"
feitas = pd.read_csv(arq) if os.path.exists(arq) else pd.DataFrame(columns=["tid", "runs", "flows"])
ja = set(feitas.tid)

for i, tid in enumerate(selected_tasks.tid):
    if tid in ja:
        continue
    t0 = time.time()
    try:
        df = openml.evaluations.list_evaluations(
            function="area_under_roc_curve", tasks=[int(tid)],
            size=1000, output_format="dataframe")
        nova = {"tid": tid, "runs": len(df), "flows": df.flow_name.nunique()}
    except Exception as e:
        print("falhou", tid, e)
        continue
    feitas = pd.concat([feitas, pd.DataFrame([nova])], ignore_index=True)
    feitas.to_csv(arq, index=False)
    print(f"{i+1}/{len(selected_tasks)}  task {tid}  {time.time()-t0:.1f}s")

In [5]:
cob = pd.read_csv(arq)
selected_tasks = selected_tasks.drop(columns=["runs", "flows"], errors="ignore").merge(cob, on="tid")

print(len(selected_tasks), "tasks com cobertura medida")
print(selected_tasks[["tid", "name", "runs", "flows"]].sort_values("flows").head(15))
print()
print((selected_tasks.flows >= 30).sum(), "tasks com >= 30 flows")

60 tasks com cobertura medida
       tid                                    name  runs  flows
49  125922                                 texture  1000     13
48  125920                           dresses-sales  1000     14
54  146821                                     car  1000     30
50  146800                             MiceProtein  1000     32
51  146817                      steel-plates-fault  1000     32
58  167140                                     dna  1000     32
53  146820                                    wilt  1000     33
52  146819        climate-model-simulation-crashes  1000     33
59  167141                                   churn  1000     33
55  146822                                 segment  1000     33
56  146824                             mfeat-pixel  1000     33
57  167119  jungle_chess_2pcs_raw_endgame_complete  1000     35
44   14952                        PhishingWebsites  1000     62
45   14954                          cylinder-bands  1000     64
36    9960

In [6]:
print(selected_tasks.flows.describe())

count     60.000000
mean     114.583333
std       56.380455
min       13.000000
25%       91.000000
50%      107.500000
75%      173.250000
max      193.000000
Name: flows, dtype: float64


In [7]:
import time
tid = 3
t0 = time.time()
partes, offset = [], 0
while offset < 20000:                     #trava de segurança
    try:
        df = openml.evaluations.list_evaluations(
            function="area_under_roc_curve", tasks=[tid],
            size=1000, offset=offset, output_format="dataframe")
    except openml.exceptions.OpenMLServerNoResult:
        break
    if len(df) == 0:
        break
    partes.append(df)
    offset += 1000
    print(offset, f"{time.time()-t0:.0f}s")
    if len(df) < 1000:
        break

tot = pd.concat(partes, ignore_index=True)
print(len(tot), "runs,", tot.flow_name.nunique(), "flows,", f"{time.time()-t0:.0f}s")

1000 6s
2000 12s
3000 19s
4000 25s
5000 30s
6000 34s
7000 39s
8000 43s
9000 47s
10000 50s
11000 53s
12000 56s
13000 59s
14000 64s
15000 68s
16000 73s
17000 78s
18000 82s
19000 87s
20000 92s
20000 runs, 814 flows, 92s


In [8]:
a = openml.evaluations.list_evaluations(function="area_under_roc_curve", tasks=[3],
    size=1000, sort_order="asc", output_format="dataframe")
b = openml.evaluations.list_evaluations(function="area_under_roc_curve", tasks=[3],
    size=1000, sort_order="desc", output_format="dataframe")
print(a.run_id.min(), a.run_id.max(), "|", b.run_id.min(), b.run_id.max())
print(len(set(a.flow_name) & set(b.flow_name)), "flows em comum")

1865494 10559877 | 594412 10593874
6 flows em comum


In [9]:
print("sort_order asc  :", a.value.min(), a.value.max())
print("sort_order desc :", b.value.min(), b.value.max())

print("run_id crescente?   ", tot.run_id.is_monotonic_increasing)
print("value decrescente?  ", tot.value.is_monotonic_decreasing)
print("value crescente?    ", tot.value.is_monotonic_increasing)
print(tot[["run_id", "value"]].head(10))

sort_order asc  : 0.00036608865466539387 0.499042
sort_order desc : 0.99979 0.9999199548922275
run_id crescente?    False
value decrescente?   False
value crescente?     False
   run_id     value
0      28  0.977253
1      78  0.951281
2     139  0.666985
3     217  0.666432
4     221  0.997912
5     246  0.951346
6     331  0.995569
7   25674  0.951281
8   35301  0.499042
9   54734  0.999415


In [10]:
t0 = time.time()
offset = 20000
while offset < 300_000:                       # teto de segurança maior
    try:
        df = openml.evaluations.list_evaluations(
            function="area_under_roc_curve", tasks=[3],
            size=1000, offset=offset, output_format="dataframe")
    except openml.exceptions.OpenMLServerNoResult:
        break
    if len(df) == 0:
        break
    partes.append(df)
    offset += 1000
    if (offset // 1000) % 10 == 0:
        print(offset, f"{time.time()-t0:.0f}s")
    if len(df) < 1000:
        break

tot = pd.concat(partes, ignore_index=True)
print(len(tot), "runs,", tot.flow_name.nunique(), "flows,", f"{time.time()-t0:.0f}s nesta etapa")

30000 53s
40000 104s
50000 152s
60000 200s
70000 248s
80000 299s
90000 354s
100000 408s
110000 460s
120000 514s
130000 571s
140000 629s
150000 690s
160000 752s
170000 810s
173304 runs, 927 flows, 834s nesta etapa


In [11]:
def contar_runs(tid, passo=1000):
    def existe(off):
        try:
            d = openml.evaluations.list_evaluations(
                function="area_under_roc_curve", tasks=[int(tid)],
                size=1, offset=off, output_format="dataframe")
            return len(d) > 0
        except openml.exceptions.OpenMLServerNoResult:
            return False
    lo, hi = 0, passo
    while existe(hi):
        lo, hi = hi, hi * 2
    while hi - lo > passo:
        mid = (lo + hi) // 2
        if existe(mid):
            lo = mid
        else:
            hi = mid
    return lo

print(3, contar_runs(3))                      #deve dar ~173.000: serve de teste da função
for t in selected_tasks.tid.sample(5, random_state=1):
    print(t, contar_runs(t))

3 173000
9977 32000
9985 18000
11 24000
125920 18000
146800 9000


In [12]:
print(len(selected_tasks), "tasks com cobertura medida")

60 tasks com cobertura medida


In [13]:
import os, time

arq_c = "../data/contagem_runs.csv"     # use o mesmo prefixo de caminho que funcionou antes
feitas_c = pd.read_csv(arq_c) if os.path.exists(arq_c) else pd.DataFrame(columns=["tid", "runs_aprox"])
ja_c = set(feitas_c.tid)

for i, tid in enumerate(selected_tasks.tid):
    if tid in ja_c:
        continue
    t0 = time.time()
    try:
        n = contar_runs(tid)
    except Exception as e:
        print("falhou", tid, e)
        continue
    feitas_c = pd.concat([feitas_c, pd.DataFrame([{"tid": tid, "runs_aprox": n}])], ignore_index=True)
    feitas_c.to_csv(arq_c, index=False)
    print(f"{i+1}/{len(selected_tasks)}  task {tid}  ~{n} runs  {time.time()-t0:.0f}s")

In [14]:
c = pd.read_csv(arq_c)
print(c.runs_aprox.describe())
print()
print(c.runs_aprox.sum(), "runs no total")
print()
print(c.sort_values("runs_aprox", ascending=False).head(10))

count        60.000000
mean      59916.666667
std       81702.082140
min        4000.000000
25%       18000.000000
50%       21000.000000
75%       95250.000000
max      415000.000000
Name: runs_aprox, dtype: float64

3595000 runs no total

      tid  runs_aprox
12     31      415000
43  10101      385000
29   3913      173000
0       3      173000
30   3917      158000
35   9957      149000
33   9946      147000
31   3918      146000
27   3903      142000
14     37      131000


In [15]:
import pandas as pd, glob

arqs = sorted(glob.glob("../data/runs/task_167140/pagina_*.csv.gz"))
df = pd.concat(pd.read_csv(a) for a in arqs)
print(len(df), "linhas,", df.run_id.nunique(), "run_id distintos")
print(df.head())
print(df.value.describe())

fl = pd.read_csv("../data/flows.csv")
print(len(fl), "flows na tabela;", df.flow_id.isin(fl.flow_id).mean(), "dos runs têm flow na tabela")

4957 linhas, 4957 run_id distintos
    run_id  setup_id  flow_id     value          upload_time
0  9199772   7130140     8456  0.986947  2018-05-03 21:14:54
1  9199845   7130139     8455  0.764915  2018-05-03 22:57:07
2  9202086   7131884     8330  0.989111  2018-05-30 17:43:36
3  9202092   7131890     8330  0.994185  2018-05-30 17:44:20
4  9202096   7131894     8330  0.994044  2018-05-30 17:44:32
count    4957.000000
mean        0.892367
std         0.165322
min         0.498363
25%         0.937513
50%         0.980153
75%         0.990912
max         0.995023
Name: value, dtype: float64
2794 flows na tabela; 1.0 dos runs têm flow na tabela


In [16]:
import glob, os

cont = pd.read_csv("../data/contagem_runs.csv").set_index("tid").runs_aprox
res = []
for pasta in sorted(glob.glob("../data/runs/task_*")):
    if not os.path.exists(os.path.join(pasta, "COMPLETA")):
        continue
    tid = int(os.path.basename(pasta).replace("task_", ""))
    arqs = sorted(glob.glob(os.path.join(pasta, "pagina_*.csv.gz")))
    d = pd.concat(pd.read_csv(a, usecols=["run_id", "value"]) for a in arqs)
    res.append({"tid": tid, "linhas": len(d), "distintos": d.run_id.nunique(),
                "esperado": cont.get(tid),
                "pct_abaixo_05": (d.value < 0.5).mean(), "pct_igual_05": (d.value == 0.5).mean()})

r = pd.DataFrame(res)
r["dif"] = r.linhas - r.esperado
print(len(r), "tasks completas;", r.linhas.sum(), "runs no total")
print("tasks com run_id repetido:", (r.linhas != r.distintos).sum())
print("tasks fora da faixa esperada:", (~r.dif.between(1, 1000)).sum())
print(r[~r.dif.between(1, 1000)][["tid", "linhas", "esperado", "dif"]])
print("AUC < 0,5 (média entre tasks):", round(r.pct_abaixo_05.mean() * 100, 2), "%")
print("AUC = 0,5 (média entre tasks):", round(r.pct_igual_05.mean() * 100, 2), "%")

60 tasks completas; 3624525 runs no total
tasks com run_id repetido: 0
tasks fora da faixa esperada: 0
Empty DataFrame
Columns: [tid, linhas, esperado, dif]
Index: []
AUC < 0,5 (média entre tasks): 11.52 %
AUC = 0,5 (média entre tasks): 5.34 %


## 2.Integridade e limpeza

In [17]:
from pathlib import Path
import collections

RUNS = Path("../data/runs")
pastas = sorted([p for p in RUNS.iterdir() if p.is_dir()], key=lambda p: int(p.name.split("_")[1]))

completas = [p for p in pastas if (p / "COMPLETA").exists()]
print("pastas de task:", len(pastas), "| com COMPLETA:", len(completas))
print("sem COMPLETA:", [p.name for p in pastas if p not in completas])

exts = collections.Counter(f.suffix for p in pastas for f in p.iterdir() if f.name.startswith("pagina_"))
print("extensões das páginas:", dict(exts))

# exemplo: primeira página da task_3
pag = sorted(f for f in (RUNS / "task_3").iterdir() if f.name.startswith("pagina_"))
print(len(pag), "páginas na task_3 | primeira:", pag[0].name)
print((RUNS / "task_3" / "COMPLETA").read_text()[:300])   # pode ter metadados úteis

pastas de task: 60 | com COMPLETA: 60
sem COMPLETA: []
extensões das páginas: {'.gz': 3655}
174 páginas na task_3 | primeira: pagina_0000000.csv.gz



In [18]:
import pandas as pd
amostra = pd.read_parquet(pag[0]) if pag[0].suffix == ".parquet" else pd.read_csv(pag[0])
print(amostra.shape, amostra.columns.tolist())
print(amostra.head(3))

(1000, 5) ['run_id', 'setup_id', 'flow_id', 'value', 'upload_time']
   run_id  setup_id  flow_id     value          upload_time
0      28        18       77  0.977253  2014-04-06 23:47:04
1      78         3       58  0.951281  2014-04-07 00:03:46
2     139         2       57  0.666985  2014-04-07 00:46:29


In [19]:
import time
from pathlib import Path
import numpy as np
import pandas as pd

RUNS = Path("../data/runs")
pastas = sorted([p for p in RUNS.iterdir() if p.is_dir()], key=lambda p: int(p.name.split("_")[1]))

t0 = time.time()
partes = []
for p in pastas:
    task_id = int(p.name.split("_")[1])
    paginas = sorted(p.glob("pagina_*.csv.gz"))
    d = pd.concat([pd.read_csv(f) for f in paginas], ignore_index=True)
    d.insert(0, "task_id", task_id)
    partes.append(d)

df = pd.concat(partes, ignore_index=True)
print(f"{len(df)} runs em {df.task_id.nunique()} tasks | {time.time()-t0:.0f}s")
print(df.dtypes)

3624525 runs em 60 tasks | 26s
task_id          int64
run_id           int64
setup_id         int64
flow_id          int64
value          float64
upload_time        str
dtype: object


In [20]:
T, R, F, V = "task_id", "run_id", "flow_id", "value"

print("tasks:", df[T].nunique(), "(esperado 60)")
print("duplicados (task, run_id):", df.duplicated([T, R]).sum())
print("run_id repetido entre tasks:", df.duplicated(R).sum() - df.duplicated([T, R]).sum())
print("colunas com nulos:\n", df.isna().sum())

baixado = df.groupby(T).size().rename("baixados")
est = pd.read_csv("../data/contagem_runs.csv")
print(est.head(3))

tasks: 60 (esperado 60)
duplicados (task, run_id): 0
run_id repetido entre tasks: 0
colunas com nulos:
 task_id        0
run_id         0
setup_id       0
flow_id        0
value          0
upload_time    0
dtype: int64
   tid  runs_aprox
0    3      173000
1    6       34000
2   11       24000


In [21]:
g = df.groupby(T)[V]
perfil = pd.DataFrame({
    "n": g.size(),
    "pct_nan": g.apply(lambda s: s.isna().mean() * 100),
    "pct_fora_0_1": g.apply(lambda s: ((s < 0) | (s > 1)).mean() * 100),
    "pct_igual_0.5": g.apply(lambda s: (s == 0.5).mean() * 100),
    "pct_menor_0.5": g.apply(lambda s: (s < 0.5).mean() * 100),
})
print(perfil.describe().round(2))
print(perfil.sort_values("pct_menor_0.5", ascending=False).head(10).round(2))

               n  pct_nan  pct_fora_0_1  pct_igual_0.5  pct_menor_0.5
count      60.00     60.0          60.0          60.00          60.00
mean    60408.75      0.0           0.0           5.34          11.52
std     81682.97      0.0           0.0           6.31           9.20
min      4957.00      0.0           0.0           0.04           0.21
25%     18766.50      0.0           0.0           1.28           2.51
50%     21501.00      0.0           0.0           2.25          10.90
75%     95716.75      0.0           0.0           8.17          16.42
max    415117.00      0.0           0.0          31.56          36.07
              n  pct_nan  pct_fora_0_1  pct_igual_0.5  pct_menor_0.5
task_id                                                             
3903     142734      0.0           0.0           1.74          36.07
10101    385696      0.0           0.0           1.35          29.05
3917     158237      0.0           0.0           1.47          28.51
14965     40166      0.0 

In [22]:
#baixado x estimado (a estimativa é arredondada em ±1000)
cmp = est.rename(columns={"tid": "task_id"}).set_index("task_id").join(baixado)
cmp["dif"] = cmp["baixados"] - cmp["runs_aprox"]
print("tasks com |dif| > 1000:", (cmp["dif"].abs() > 1000).sum())
print(cmp[cmp["dif"].abs() > 1000])

# diagnóstico dos suspeitos: mediana da AUC e proporções, por task
diag = perfil.join(df.groupby(T)[V].median().rename("mediana_auc"))
print(diag.sort_values("pct_igual_0.5", ascending=False).head(5).round(2))
print(diag.sort_values("pct_menor_0.5", ascending=False).head(10).round(2))

tasks com |dif| > 1000: 0
Empty DataFrame
Columns: [runs_aprox, baixados, dif]
Index: []
              n  pct_nan  pct_fora_0_1  pct_igual_0.5  pct_menor_0.5  \
task_id                                                                
31       415117      0.0           0.0          31.56           1.66   
146819     8792      0.0           0.0          19.29           0.96   
125920    18972      0.0           0.0          16.90           6.17   
22        20027      0.0           0.0          16.19           2.51   
146821     7166      0.0           0.0          15.88           1.76   

         mediana_auc  
task_id               
31              0.75  
146819          0.89  
125920          0.57  
22              0.90  
146821          0.99  
              n  pct_nan  pct_fora_0_1  pct_igual_0.5  pct_menor_0.5  \
task_id                                                                
3903     142734      0.0           0.0           1.74          36.07   
10101    385696      0.0     

In [23]:
n0 = len(df)
d = df[df[V] != 0.5]
print(f"antes {n0} | depois {len(d)} | removidos {n0 - len(d)} ({(n0 - len(d)) / n0 * 100:.1f}%)")

restantes = d.groupby(T).size()
print("menor n por task após limpeza:", restantes.min(), "| task:", restantes.idxmin())

d.to_parquet("../data/runs_limpos.parquet", index=False)
df.to_parquet("../data/runs_consolidado.parquet", index=False)

antes 3624525 | depois 3406019 | removidos 218506 (6.0%)
menor n por task após limpeza: 4714 | task: 167141


In [24]:
for t in [3903, 3902, 14965, 219, 9952, 10101]:
    s = df[df[T] == t]
    b = s[s[V] < 0.5]
    fl = s.groupby(F)[V].agg(n="size", med="median")
    print(f"task {t}: {len(b)} runs < 0,5 | quantis 10/50/90 deles: {b[V].quantile([.1, .5, .9]).round(3).tolist()}"
          f" | flows com ao menos 1 run < 0,5: {b[F].nunique()} de {s[F].nunique()}"
          f" | flows com mediana < 0,5: {(fl.med < 0.5).sum()}"
          f" | 3 flows com mais runs < 0,5 (fração): {b[F].value_counts(normalize=True).head(3).round(2).to_dict()}")

task 3903: 51479 runs < 0,5 | quantis 10/50/90 deles: [0.499, 0.499, 0.499] | flows com ao menos 1 run < 0,5: 85 de 509 | flows com mediana < 0,5: 56 | 3 flows com mais runs < 0,5 (fração): {6766: 0.45, 5636: 0.38, 5527: 0.05}
task 3902: 27101 runs < 0,5 | quantis 10/50/90 deles: [0.496, 0.496, 0.496] | flows com ao menos 1 run < 0,5: 65 de 512 | flows com mediana < 0,5: 35 | 3 flows com mais runs < 0,5 (fração): {5636: 0.55, 6766: 0.13, 5859: 0.12}
task 14965: 10676 runs < 0,5 | quantis 10/50/90 deles: [0.5, 0.5, 0.5] | flows com ao menos 1 run < 0,5: 31 de 266 | flows com mediana < 0,5: 11 | 3 flows com mais runs < 0,5 (fração): {5636: 0.36, 5859: 0.18, 6766: 0.17}
task 219: 19225 runs < 0,5 | quantis 10/50/90 deles: [0.5, 0.5, 0.5] | flows com ao menos 1 run < 0,5: 44 de 620 | flows com mediana < 0,5: 27 | 3 flows com mais runs < 0,5 (fração): {6766: 0.4, 5636: 0.33, 5970: 0.07}
task 9952: 29536 runs < 0,5 | quantis 10/50/90 deles: [0.499, 0.499, 0.499] | flows com ao menos 1 run < 

In [25]:
import openml

sus = [6766, 5636, 5859, 5970, 6767]
for f in sus:
    fl = openml.flows.get_flow(f)
    s = df[df[F] == f][V]
    print(f, "|", fl.name, "|", len(s), "runs |",
          "frac < 0,5:", round((s < 0.5).mean(), 3),
          "| frac == 0,5:", round((s == 0.5).mean(), 3),
          "| quantis 5/50/95:", s.quantile([.05, .5, .95]).round(4).tolist())

6766 | mlr.classif.glmnet | 403548 runs | frac < 0,5: 0.48 | frac == 0,5: 0.085 | quantis 5/50/95: [0.4871, 0.5, 0.9604]
5636 | mlr.classif.rpart | 407964 runs | frac < 0,5: 0.615 | frac == 0,5: 0.184 | quantis 5/50/95: [0.4871, 0.499, 0.8589]
5859 | mlr.classif.rpart | 38242 runs | frac < 0,5: 0.645 | frac == 0,5: 0.074 | quantis 5/50/95: [0.4871, 0.4993, 0.8694]
5970 | mlr.classif.glmnet | 38165 runs | frac < 0,5: 0.53 | frac == 0,5: 0.026 | quantis 5/50/95: [0.4933, 0.4999, 0.9933]
6767 | mlr.classif.xgboost | 357384 runs | frac < 0,5: 0.031 | frac == 0,5: 0.079 | quantis 5/50/95: [0.5, 0.7852, 0.9963]


In [26]:
fl = df.groupby(F)[V].agg(n="size", mediana="median", q90=lambda s: s.quantile(.9))
deg = fl[(fl.n >= 50) & (fl.q90 <= 0.505)].sort_values("n", ascending=False)
print(len(deg), "flows com >= 50 runs e 90% dos runs <= 0,505")
print(deg.head(15).round(4))

5 flows com >= 50 runs e 90% dos runs <= 0,505
           n  mediana     q90
flow_id                      
6602     110   0.4970  0.4999
1718     109   0.4987  0.5000
6613     100   0.4990  0.4999
1069      79   0.4987  0.5000
17701     61   0.4946  0.4967


In [27]:
# distribuição global perto de 0,5, em faixas de 0,002
near = df[(df[V] >= 0.48) & (df[V] <= 0.52)]
print(near.groupby(pd.cut(near[V], np.arange(0.48, 0.5201, 0.002)), observed=True).size())

# nas tasks suspeitas: os valores mais repetidos (4 casas)
for t in [3903, 3902, 14965, 9952]:
    s = df[(df[T] == t) & (df[V] < 0.5)][V].round(4)
    print(t, s.value_counts().head(4).to_dict())

value
(0.48, 0.482]        996
(0.482, 0.484]      1012
(0.484, 0.486]     32954
(0.486, 0.488]     40138
(0.488, 0.49]       2265
(0.49, 0.492]      27304
(0.492, 0.494]     29852
(0.494, 0.496]    222290
(0.496, 0.498]     40262
(0.498, 0.5]      448320
(0.5, 0.502]       27704
(0.502, 0.504]      6989
(0.504, 0.506]      6202
(0.506, 0.508]      3068
(0.508, 0.51]       4092
(0.51, 0.512]       3412
(0.512, 0.514]      4397
(0.514, 0.516]      4532
(0.516, 0.518]      3281
(0.518, 0.52]       3428
dtype: int64
3903 {0.4993: 47315, 0.4998: 262, 0.4996: 243, 0.4999: 186}
3902 {0.4955: 25162, 0.4966: 304, 0.4961: 297, 0.4972: 196}
14965 {0.4999: 9935, 0.5: 581, 0.4998: 8, 0.4997: 7}
9952 {0.499: 28024, 0.4996: 428, 0.4992: 377, 0.4997: 107}


In [28]:
b = (df[V] >= 0.49) & (df[V] <= 0.51)
st = df.assign(b=b).groupby("setup_id").agg(n=("b", "size"), frac=("b", "mean"))
st = st[st.n >= 5]
print(st.frac.describe().round(3))
print((st.frac > 0.9).sum(), "setups com >= 5 runs e > 90% na faixa [0,49; 0,51]")
print(len(st), "setups com >= 5 runs no total")

count    6172.000
mean        0.032
std         0.127
min         0.000
25%         0.000
50%         0.000
75%         0.000
max         1.000
Name: frac, dtype: float64
47 setups com >= 5 runs e > 90% na faixa [0,49; 0,51]
6172 setups com >= 5 runs no total


In [29]:
LO, HI = 0.47, 0.53

cand = df[(df[V] >= LO) & (df[V] <= HI)].assign(v4=lambda x: x[V].round(4))
cnt = cand.groupby([T, "v4"]).size().rename("n").reset_index()
cnt = cnt.join(df.groupby(T).size().rename("n_task"), on=T)
cnt["frac"] = cnt["n"] / cnt["n_task"]

const = cnt[(cnt["n"] >= 200) & (cnt["frac"] >= 0.005)]
print(len(const), "pares (task, valor) repetidos em", const[T].nunique(), "tasks")
print(const.sort_values("n", ascending=False).head(15).round(4))

mask = pd.Series(False, index=df.index)
for t, g in const.groupby(T):
    idx = df.index[df[T] == t]
    vals = df.loc[idx, V].to_numpy()
    m = np.zeros(len(vals), bool)
    for c in g["v4"]:
        m |= np.abs(vals - c) <= 0.0005
    mask.loc[idx] = m

print(f"\nremovidos: {mask.sum()} ({mask.mean() * 100:.1f}% do total)")
d2 = df[~mask]

pt = pd.DataFrame({"antes": df.groupby(T).size(), "depois": d2.groupby(T).size()})
pt["pct_removido"] = (1 - pt["depois"] / pt["antes"]) * 100
print(pt["pct_removido"].describe().round(1))
print(pt.sort_values("pct_removido", ascending=False).head(8).round(1))
print("menor n por task:", pt["depois"].min(), "| task:", pt["depois"].idxmin())
print("pct < 0,5 depois da regra (por task):")
print((d2[V] < 0.5).groupby(d2[T]).mean().mul(100).describe().round(2))

127 pares (task, valor) repetidos em 54 tasks
       task_id      v4       n  n_task    frac
2964        31  0.5000  131017  415117  0.3156
12582    10101  0.4955  109236  385696  0.2832
7204      3903  0.4993   47315  142734  0.3315
8655      3917  0.4957   41207  158237  0.2604
8036      3913  0.4871   38864  173611  0.2239
9091      3918  0.4856   31440  146719  0.2143
10312     9957  0.4960   30542  149481  0.2043
10143     9952  0.4990   28024  113430  0.2471
11131     9971  0.4908   25693  122665  0.2095
3672        37  0.4970   25623  131153  0.1954
6676      3902  0.4955   25162  112333  0.2240
27           3  0.4990   21727  173304  0.1254
3951        43  0.4991   20760   99163  0.2094
9923      9946  0.4933   20716  147634  0.1403
4818       219  0.4999   18179   77391  0.2349

removidos: 823307 (22.7% do total)
count    60.0
mean     15.6
std       8.3
min       0.0
25%      11.6
50%      14.7
75%      22.0
max      37.0
Name: pct_removido, dtype: float64
          antes  de

In [30]:
mask2 = mask | (df[V] == 0.5)
print("extra por AUC==0,5 fora da regra:", (mask2 & ~mask).sum())
d2 = df[~mask2]
print(len(df), "->", len(d2), f"({mask2.mean() * 100:.1f}% removido)")

#pares mais fracos (perto do limiar): conferir se parecem constantes de verdade
print(const.sort_values("n").head(8).round(4))

#tasks sem nenhum par detectado e tasks com mais < 0,5 residual
sem = sorted(set(df[T].unique()) - set(const[T]))
print("tasks sem par:", sem)
res = (d2[V] < 0.5).groupby(d2[T]).mean().mul(100).sort_values(ascending=False)
print(res.head(6).round(2))

d2.to_parquet("../data/runs_limpos_v2.parquet", index=False)

extra por AUC==0,5 fora da regra: 690
3624525 -> 2800528 (22.7% removido)
       task_id      v4    n  n_task    frac
3317        32  0.4999  200   23410  0.0085
5175      2079  0.5010  202   19958  0.0101
3318        32  0.5000  212   23410  0.0091
760         12  0.5083  217   23092  0.0094
1096        15  0.4968  222   24300  0.0091
10578     9960  0.4992  228   20137  0.0113
1102        15  0.4993  232   24300  0.0095
1099        15  0.4981  236   24300  0.0097
tasks sem par: [np.int64(146800), np.int64(146817), np.int64(146822), np.int64(146824), np.int64(167119), np.int64(167140)]
task_id
3022    8.96
2079    8.14
3560    7.78
3549    6.89
53      5.89
9964    5.83
Name: value, dtype: float64


In [31]:
for t in [3022, 2079, 3560]:
    s = d2[(d2[T] == t) & (d2[V] < 0.5)][V]
    print(t, "| mediana da task:", round(d2[d2[T] == t][V].median(), 3),
          "| runs < 0,5:", len(s),
          "| valores mais repetidos (3 casas):", s.round(3).value_counts().head(3).to_dict())

3022 | mediana da task: 0.971 | runs < 0,5: 1498 | valores mais repetidos (3 casas): {0.423: 112, 0.191: 102, 0.168: 69}
2079 | mediana da task: 0.838 | runs < 0,5: 1472 | valores mais repetidos (3 casas): {0.495: 229, 0.499: 229, 0.498: 218}
3560 | mediana da task: 0.547 | runs < 0,5: 1300 | valores mais repetidos (3 casas): {0.497: 224, 0.499: 190, 0.496: 169}


In [32]:
src = d2
cand = src[(src[V] >= 0.47) & (src[V] <= 0.53)].assign(v3=lambda x: x[V].round(3))
cnt3 = cand.groupby([T, "v3"]).size().rename("n").reset_index()
cnt3 = cnt3.join(src.groupby(T).size().rename("n_task"), on=T)
const3 = cnt3[(cnt3["n"] >= 200) & (cnt3["n"] / cnt3["n_task"] >= 0.005)]
print(len(const3), "pares extras (3 casas) em", const3[T].nunique(), "tasks")

mask3 = pd.Series(False, index=src.index)
for t, g in const3.groupby(T):
    idx = src.index[src[T] == t]
    vals = src.loc[idx, V].to_numpy()
    m = np.zeros(len(vals), bool)
    for c in g["v3"]:
        m |= np.abs(vals - c) <= 0.0005
    mask3.loc[idx] = m

print("extra removidos:", mask3.sum(), f"({mask3.sum() / len(src) * 100:.2f}% do que sobrou)")
print(mask3.groupby(src[T]).sum().sort_values(ascending=False).head(8))

55 pares extras (3 casas) em 22 tasks
extra removidos: 24792 (0.89% do que sobrou)
task_id
3560    2981
31      2737
37      2437
49      1841
9971    1778
9978    1435
53      1377
3917    1283
dtype: int64


In [33]:
mask3f = mask3 & (src[T] != 3560)
d3 = src[~mask3f]

print(len(src), "->", len(d3), f"| extras removidos: {mask3f.sum()}")
print("menor n por task:", d3.groupby(T).size().min())
for t in [2079, 3022]:
    print(t, "| % < 0,5 antes:", round((src[src[T] == t][V] < 0.5).mean() * 100, 2),
          "| depois:", round((d3[d3[T] == t][V] < 0.5).mean() * 100, 2))

d3.to_parquet("../data/runs_limpos_v3.parquet", index=False)

2800528 -> 2778717 | extras removidos: 21811
menor n por task: 4679
2079 | % < 0,5 antes: 8.14 | depois: 4.57
3022 | % < 0,5 antes: 8.96 | depois: 8.96


## 3.Famílias

In [34]:
import pandas as pd

runs = pd.read_parquet("../data/runs_limpos_v3.parquet")
print(runs.shape)
print(runs.dtypes)
print(runs.head())

flows = pd.read_csv("../data/flows.csv")
print(flows.shape)
print(flows.dtypes)
print(flows.head(10))

(2778717, 6)
task_id          int64
run_id           int64
setup_id         int64
flow_id          int64
value          float64
upload_time        str
dtype: object
   task_id  run_id  setup_id  flow_id     value          upload_time
0        3      28        18       77  0.977253  2014-04-06 23:47:04
1        3      78         3       58  0.951281  2014-04-07 00:03:46
2        3     139         2       57  0.666985  2014-04-07 00:46:29
3        3     217         7       62  0.666432  2014-04-07 01:25:07
4        3     221         6       61  0.997912  2014-04-07 01:27:12
(2794, 2)
flow_id      int64
flow_name      str
dtype: object
   flow_id              flow_name
0       56          weka.ZeroR(1)
1       57           weka.OneR(1)
2       58     weka.NaiveBayes(1)
3       59           weka.JRip(1)
4       60            weka.J48(1)
5       61        weka.REPTree(1)
6       62  weka.DecisionStump(1)
7       63  weka.HoeffdingTree(1)
8       64     weka.RandomTree(1)
9       65   weka.R

In [35]:
df = runs.merge(flows, on="flow_id", how="left")
print("flows sem nome:", df["flow_name"].isna().sum())

# biblioteca = prefixo antes do primeiro ponto
df["biblioteca"] = df["flow_name"].str.extract(r"^([A-Za-z0-9_]+)\.")[0]

# 1) visão por biblioteca
por_lib = (df.groupby("biblioteca")
             .agg(runs=("run_id", "size"),
                  flows=("flow_id", "nunique"),
                  tasks=("task_id", "nunique"))
             .sort_values("runs", ascending=False))
por_lib["pct_runs"] = (100 * por_lib["runs"] / por_lib["runs"].sum()).round(2)
print(por_lib)

# 2) bibliotecas que não casaram com o padrão
print(df.loc[df["biblioteca"].isna(), "flow_name"].drop_duplicates().head(30))

# 3) top flows por volume
por_flow = (df.groupby(["biblioteca", "flow_id", "flow_name"])
              .agg(runs=("run_id", "size"), tasks=("task_id", "nunique"))
              .sort_values("runs", ascending=False))
print(por_flow.head(40))

flows sem nome: 0
                       runs  flows  tasks  pct_runs
biblioteca                                         
mlr                 1823370    425     58     65.62
sklearn              914732   1194     60     32.92
weka                  38105    709     60      1.37
classif                1811    190     46      0.07
openmlpimp              101      1     48      0.00
arbok                    81     16     40      0.00
keras                    78     29     11      0.00
pipelineopt              44      1     41      0.00
mxnet                    42     16      1      0.00
bsplitz                  24      1     12      0.00
onnx                     22      7      1      0.00
rm                       20      9      2      0.00
fractal                  19      7      4      0.00
torch                    17     17      2      0.00
__main__                 10      6      3      0.00
tensorflow               10      3      1      0.00
optimus                  10      8      2     

In [36]:
#família mlr: tira "mlr.classif." e a versão "(N)"
m = df["biblioteca"].eq("mlr")
df.loc[m, "familia"] = ("mlr:" + df.loc[m, "flow_name"]
    .str.replace(r"^mlr\.classif\.", "", regex=True)
    .str.replace(r"\(\d+\)$", "", regex=True))

cob_mlr = (df[m].groupby("familia")
             .agg(runs=("run_id", "size"),
                  flows=("flow_id", "nunique"),
                  tasks=("task_id", "nunique"))
             .sort_values("tasks", ascending=False))
print(cob_mlr)
print("bases cobertas por pelo menos uma família mlr:", df[m]["task_id"].nunique())

                                      runs  flows  tasks
familia                                                 
mlr:rpart                            93276     22     57
mlr:naiveBayes.preproc.preproc          46      1     46
mlr:naiveBayes.preproc                 126      5     45
mlr:gbm.preproc                         76      2     44
mlr:C50.preproc                         79     43     44
...                                    ...    ...    ...
mlr:gausspr                              1      1      1
mlr:naiveBayes.imputed.preproc           1      1      1
mlr:ksvm                                 1      1      1
mlr:randomForestSRC.imputed.preproc      1      1      1
mlr:rpart.tuned                          3      3      1

[93 rows x 3 columns]
bases cobertas por pelo menos uma família mlr: 58


In [37]:
pd.set_option("display.max_colwidth", None)
top_sk = (df[df.biblioteca == "sklearn"].groupby(["flow_id", "flow_name"])
            .size().sort_values(ascending=False).head(10))
for (fid, nome), n in top_sk.items():
    print(fid, n, nome, "\n")

8817 104800 sklearn.pipeline.Pipeline(columntransformer=sklearn.compose._column_transformer.ColumnTransformer(numeric=sklearn.pipeline.Pipeline(imputer=sklearn.preprocessing.imputation.Imputer,standardscaler=sklearn.preprocessing.data.StandardScaler),nominal=sklearn.pipeline.Pipeline(simpleimputer=sklearn.impute.SimpleImputer,onehotencoder=sklearn.preprocessing._encoders.OneHotEncoder)),variancethreshold=sklearn.feature_selection.variance_threshold.VarianceThreshold,svc=sklearn.svm.classes.SVC)(1) 

18601 71721 sklearn.pipeline.Pipeline(simpleimputer=sklearn.impute._base.SimpleImputer,standardscaler=sklearn.preprocessing.data.StandardScaler,logisticregression=sklearn.linear_model.logistic.LogisticRegression)(2) 

6969 65842 sklearn.pipeline.Pipeline(imputation=openmlstudy14.preprocessing.ConditionalImputer,hotencoding=sklearn.preprocessing.data.OneHotEncoder,variencethreshold=sklearn.feature_selection.variance_threshold.VarianceThreshold,classifier=sklearn.ensemble.forest.RandomForestC

In [38]:
import re

def ultimo_passo(s):
    s = re.sub(r"\(\d+\)$", "", s)                  #tira a versão
    if not s.startswith("sklearn.pipeline.Pipeline("):
        return s.split("(")[0].split(".")[-1]        #classe mais externa
    inner = s[len("sklearn.pipeline.Pipeline("):-1]
    partes, depth, cur = [], 0, ""
    for ch in inner:
        depth += (ch == "(") - (ch == ")")
        if ch == "," and depth == 0:
            partes.append(cur); cur = ""
        else:
            cur += ch
    partes.append(cur)
    ultimo = partes[-1].split("=", 1)[1]
    return ultimo_passo(ultimo) if ultimo.startswith("sklearn.pipeline.Pipeline(") \
           else ultimo.split("(")[0].split(".")[-1]

# --- sklearn
sk = df["biblioteca"].eq("sklearn")
df.loc[sk, "familia"] = "sklearn:" + df.loc[sk, "flow_name"].map(ultimo_passo)

# --- mlr: normaliza sufixos
mlr_ = df["biblioteca"].eq("mlr")
df.loc[mlr_, "familia"] = "mlr:" + (df.loc[mlr_, "flow_name"]
    .str.replace(r"^mlr\.classif\.", "", regex=True)
    .str.replace(r"\(\d+\)$", "", regex=True)
    .str.replace(r"\.(preproc|imputed|tuned)", "", regex=True))

# --- weka: com e sem prefixo, ignorando maiúsculas
wk_sem = df["biblioteca"].isna() & ~df["flow_name"].str.startswith("automlbenchmark_")
wk = df["biblioteca"].eq("weka") | wk_sem
df.loc[wk, "familia"] = "weka:" + (df.loc[wk, "flow_name"]
    .str.replace(r"^weka\.", "", regex=True)
    .str.replace(r"\(\d+\)$", "", regex=True)
    .str.split(".").str[-1].str.lower())

# --- cobertura
cob = (df.dropna(subset=["familia"]).groupby("familia")
         .agg(runs=("run_id", "size"), flows=("flow_id", "nunique"),
              tasks=("task_id", "nunique"))
         .sort_values("tasks", ascending=False))
print(len(cob), "famílias;", (cob.tasks >= 42).sum(), "com >= 42 bases")
print(cob.head(40))

1018 famílias; 131 com >= 42 bases
                                                    runs  flows  tasks
familia                                                               
sklearn:LogisticRegression                         81308     56     60
sklearn:HalvingRandomSearchCV                       1039      4     60
sklearn:HistGradientBoostingClassifier             29411     30     60
sklearn:KNeighborsClassifier                        1049     40     60
sklearn:FKCEigenPro                                50275      1     60
sklearn:ExtraTreesClassifier                        1398     13     60
sklearn:GradientBoostingClassifier                 54487     30     60
sklearn:GaussianNB                                   359     18     60
sklearn:DecisionTreeClassifier                     73973    143     60
sklearn:BernoulliNB                                 5827      6     60
sklearn:AdaBoostClassifier                         29939     40     60
sklearn:MLPClassifier                     

In [39]:
#weka: tira parênteses finais e normaliza wrappers
def limpa_weka(nome):
    n = re.sub(r"^weka\.", "", nome)
    n = re.sub(r"\(\d+\)$", "", n)
    n = n.rstrip(")")
    return n.split(".")[-1].lower()

df.loc[wk, "familia"] = "weka:" + df.loc[wk, "flow_name"].map(limpa_weka)

# sklearn: junta FKC_EigenPro com FKCEigenPro
df["familia"] = df["familia"].str.replace("FKC_EigenPro", "FKCEigenPro", regex=False)

In [40]:
w = df[df.familia.str.startswith("weka:", na=False)]
nomes = (w.groupby("familia").agg(runs=("run_id","size"), tasks=("task_id","nunique"))
           .sort_values("tasks", ascending=False))
print(nomes.head(60))
print(df.loc[df.familia.str.contains("SearchCV", na=False), "flow_name"].drop_duplicates().head(5).tolist())

                                                            runs  tasks
familia                                                                
weka:reptree                                                 292     60
weka:j48                                                    1764     60
weka:logistic                                                222     60
weka:randomforest                                           1609     59
weka:jrip                                                    190     50
weka:filteredclassifier_multisearch_logistic                  48     48
weka:filteredclassifier_naivebayes                            70     48
weka:filteredclassifier_multisearch_ibk                       48     48
weka:filteredclassifier_multisearch_randomforest              47     47
weka:bagging_j48                                             293     46
weka:bagging_jrip                                            194     46
weka:attributeselectedclassifier_multilayerperceptron        173

In [41]:
BUSCAS = {"RandomizedSearchCV", "GridSearchCV",
          "HalvingRandomSearchCV", "HalvingGridSearchCV"}

def split_top(s):
    partes, depth, cur = [], 0, ""
    for ch in s:
        depth += (ch == "(") - (ch == ")")
        if ch == "," and depth == 0:
            partes.append(cur); cur = ""
        else:
            cur += ch
    partes.append(cur)
    return partes

def familia_sk(s):
    s = re.sub(r"\(\d+\)$", "", s)
    head = s.split("(")[0]
    cls = head.split(".")[-1]
    args = s[len(head) + 1:-1] if "(" in s else ""
    try:
        if cls in BUSCAS:
            return familia_sk(split_top(args)[0].split("=", 1)[1])
        if cls == "Pipeline":
            return familia_sk(split_top(args)[-1].split("=", 1)[1])
    except IndexError:
        pass
    return cls.replace("FKC_EigenPro", "FKCEigenPro")

WRAP_BUSCA = {"filteredclassifier", "multisearch", "multischeme"}

def familia_weka(nome):
    n = re.sub(r"^weka\.", "", nome)
    n = re.sub(r"\(\d+\)$", "", n).rstrip(")").split(".")[-1].lower()
    toks = [t for t in n.split("_") if t]
    # remove wrappers de busca/filtro do começo; o que sobra começa pela família externa
    while toks and toks[0] in WRAP_BUSCA and len(toks) > 1:
        toks = toks[1:]
    return toks[0]

sk = df["biblioteca"].eq("sklearn")
df.loc[sk, "familia"] = "sklearn:" + df.loc[sk, "flow_name"].map(familia_sk)

wk = df["biblioteca"].eq("weka") | (df["biblioteca"].isna()
      & ~df["flow_name"].str.startswith("automlbenchmark_"))
df.loc[wk, "familia"] = "weka:" + df.loc[wk, "flow_name"].map(familia_weka)

cob = (df.dropna(subset=["familia"]).groupby("familia")
         .agg(runs=("run_id", "size"), flows=("flow_id", "nunique"),
              tasks=("task_id", "nunique"))
         .sort_values("tasks", ascending=False))
print(len(cob), "famílias;", (cob.tasks >= 42).sum(), "com >= 42 bases")
print(cob.head(40))
print(cob.loc[cob.index.str.contains("ranger|xgboost")])

661 famílias; 89 com >= 42 bases
                                          runs  flows  tasks
familia                                                     
sklearn:AdaBoostClassifier               30845     41     60
sklearn:ExtraTreesClassifier              1404     15     60
weka:reptree                               360      9     60
sklearn:BernoulliNB                       5827      6     60
sklearn:FKCEigenPro                      50604      3     60
weka:logistic                              270      9     60
sklearn:GaussianNB                         359     18     60
sklearn:GradientBoostingClassifier       54487     30     60
sklearn:HistGradientBoostingClassifier   31126     38     60
sklearn:KNeighborsClassifier              1049     40     60
sklearn:SVC                             348056     94     60
sklearn:SGDClassifier                      409     28     60
sklearn:RandomForestClassifier          233325    128     60
sklearn:LogisticRegression               81319     5

In [42]:
mlr_ = df["biblioteca"].eq("mlr")
df.loc[mlr_, "familia"] = df.loc[mlr_, "familia"].str.replace(r"\.dummied$", "", regex=True)

cob = (df.dropna(subset=["familia"]).groupby("familia")
         .agg(runs=("run_id", "size"), flows=("flow_id", "nunique"),
              tasks=("task_id", "nunique"))
         .sort_values("tasks", ascending=False))

print(cob.loc[cob.index.str.contains("ranger|xgboost")])
for lim in (36, 39, 42, 45, 48):
    print(f">= {lim} bases ({lim/60:.0%}):", (cob.tasks >= lim).sum(), "famílias")
print(cob.groupby(cob.index.str.split(":").str[0])["tasks"].apply(lambda s: (s >= 42).sum()))

               runs  flows  tasks
familia                          
mlr:ranger   748286     52     45
mlr:xgboost  360955      8     22
>= 36 bases (60%): 110 famílias
>= 39 bases (65%): 96 famílias
>= 42 bases (70%): 90 famílias
>= 45 bases (75%): 63 famílias
>= 48 bases (80%): 27 famílias
familia
mlr         9
sklearn    16
weka       65
Name: tasks, dtype: int64


In [43]:
df.to_parquet("../data/runs_v3_familias.parquet", index=False)

In [44]:
df = pd.read_parquet("../data/runs_v3_familias.parquet")
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(2778717, 9)
['task_id', 'run_id', 'setup_id', 'flow_id', 'value', 'upload_time', 'flow_name', 'biblioteca', 'familia']
   task_id  run_id  setup_id  flow_id     value          upload_time  \
0        3      28        18       77  0.977253  2014-04-06 23:47:04   
1        3      78         3       58  0.951281  2014-04-07 00:03:46   
2        3     139         2       57  0.666985  2014-04-07 00:46:29   

                          flow_name biblioteca          familia  
0  weka.LogitBoost_DecisionStump(1)       weka  weka:logitboost  
1                weka.NaiveBayes(1)       weka  weka:naivebayes  
2                      weka.OneR(1)       weka        weka:oner  


In [45]:
#2a. Famílias sem valor e quantas famílias existem no df
print("NaN em familia:", df["familia"].isna().sum())
print("famílias distintas (antes do corte):", df["familia"].nunique())
print(df["biblioteca"].value_counts())

#2b. Bases e runs por família
resumo = (df.groupby("familia")
            .agg(biblioteca=("biblioteca", "first"),
                 n_bases=("task_id", "nunique"),
                 n_runs=("run_id", "size"),
                 n_flows=("flow_id", "nunique"))
            .sort_values("n_bases", ascending=False))
n_tasks = df["task_id"].nunique()
print("tasks:", n_tasks, "| corte 70% =", int(0.7 * n_tasks), "bases")

#2c. Aplicar o corte e conferir se dá 90
elegiveis = resumo[resumo["n_bases"] >= 42]
print("famílias com >= 42 bases:", len(elegiveis))
print(elegiveis["biblioteca"].value_counts())

#2d. Uma família pode ter mais de uma biblioteca?
print((df.groupby("familia")["biblioteca"].nunique() > 1).sum(), "famílias com mais de uma biblioteca")

NaN em familia: 2377
famílias distintas (antes do corte): 657
biblioteca
mlr                   1823370
sklearn                914732
weka                    38105
classif                  1811
openmlpimp                101
arbok                      81
keras                      78
pipelineopt                44
mxnet                      42
bsplitz                    24
onnx                       22
rm                         20
fractal                    19
torch                      17
__main__                   10
optimus                    10
tensorflow                 10
optimus_ml                  9
onnx_ml_pb2                 9
wrapper                     7
xgboost                     5
wrappers                    4
randomsubgroups             4
lightgbm                    3
mlr3                        3
extra                       3
versus_full                 3
hpo                         3
autosklearn                 2
openml_estimator            2
fastsklearnfeature         

In [46]:
sem_fam = df[df["familia"].isna()]
print(sem_fam["flow_name"].value_counts().head(20))
print(sem_fam["biblioteca"].value_counts().head())

flow_name
openmlpimp.sklearn.beam_search.BeamSearchCV(estimator=sklearn.pipeline.Pipeline(imputation=openmlstudy14.preprocessing.ConditionalImputer,hotencoding=sklearn.preprocessing.data.OneHotEncoder,variencethreshold=sklearn.feature_selection.variance_threshold.VarianceThreshold,classifier=sklearn.ensemble.forest.RandomForestClassifier))(1)    101
pipelineopt.estimator.Classifier(1)                                                                                                                                                                                                                                                                                                                44
classif.randomForest(43)                                                                                                                                                                                                                                                                                          

In [47]:
df = df[df["familia"].notna()].copy()
print(df.shape)

(2776340, 9)


In [48]:
import numpy as np
cont = df.groupby(["task_id", "familia"]).size().unstack(fill_value=0)  # bases x famílias
n_bases = cont.shape[0]

linhas = []
for m in [1, 3, 5, 10]:
    presente = cont >= m
    cobertura = presente.mean()  #fração de bases por família
    for corte in [0.6, 0.65, 0.7, 0.75, 0.8]:
        fams = cobertura[cobertura >= corte].index
        sub = presente[fams]
        linhas.append({"min_runs": m, "corte": corte, "n_familias": len(fams),
                       "pct_celulas_vazias": 1 - sub.values.mean(),
                       "min_fams_por_base": sub.sum(axis=1).min()})
print(pd.DataFrame(linhas).to_string(index=False))

 min_runs  corte  n_familias  pct_celulas_vazias  min_fams_por_base
        1   0.60         110            0.223182                 18
        1   0.65          96            0.200174                 18
        1   0.70          90            0.191481                 18
        1   0.75          63            0.157407                 18
        1   0.80          27            0.052469                 18
        3   0.60          27            0.191358                  7
        3   0.65          26            0.183974                  7
        3   0.70          22            0.156061                  7
        3   0.75          19            0.135088                  7
        3   0.80          10            0.050000                  7
        5   0.60          17            0.155882                  6
        5   0.65          16            0.142708                  6
        5   0.70          15            0.130000                  6
        5   0.75          15            0.130000

In [49]:
cont_long = df.groupby(["task_id", "familia", "biblioteca"]).size().reset_index(name="n")
print(cont_long.groupby("biblioteca")["n"].describe(percentiles=[.1, .5, .9]))

#composição por biblioteca de cada cenário
presente = cont >= 3
cob = presente.mean()
fams = cob[cob >= 0.70].index
print(df.drop_duplicates("familia").set_index("familia").loc[fams, "biblioteca"].value_counts())

#a base com poucas famílias (min_fams_por_base)
print(presente[fams].sum(axis=1).sort_values().head())

             count         mean          std  min  10%  50%     90%       max
biblioteca                                                                   
mlr         1061.0  1718.539114  7879.827016  1.0  1.0  2.0  1392.0  107790.0
sklearn     1515.0   603.783498  1489.516487  1.0  1.0  2.0  1750.8   11520.0
weka        4031.0     9.452989    32.167493  1.0  1.0  1.0    15.0     363.0
biblioteca
weka       11
sklearn     9
mlr         2
Name: count, dtype: int64
task_id
167140    7
146821    7
167119    7
14954     8
167141    8
dtype: int64


In [50]:
for m in [3, 5]:
    presente = cont >= m
    cob = presente.mean()
    fams = cob[cob >= 0.70].index
    print(f"\nmin {m}:", len(fams), "famílias")
    print(sorted(fams))


min 3: 22 famílias
['mlr:naiveBayes', 'mlr:rpart', 'sklearn:AdaBoostClassifier', 'sklearn:DecisionTreeClassifier', 'sklearn:FKCEigenPro', 'sklearn:GaussianNB', 'sklearn:GradientBoostingClassifier', 'sklearn:HistGradientBoostingClassifier', 'sklearn:LogisticRegression', 'sklearn:RandomForestClassifier', 'sklearn:SVC', 'weka:adaboostm1', 'weka:attributeselectedclassifier', 'weka:bagging', 'weka:ibk', 'weka:j48', 'weka:logistic', 'weka:lwl', 'weka:multilayerperceptron', 'weka:naivebayes', 'weka:randomforest', 'weka:reptree']

min 5: 15 famílias
['sklearn:AdaBoostClassifier', 'sklearn:DecisionTreeClassifier', 'sklearn:FKCEigenPro', 'sklearn:GradientBoostingClassifier', 'sklearn:HistGradientBoostingClassifier', 'sklearn:LogisticRegression', 'sklearn:RandomForestClassifier', 'sklearn:SVC', 'weka:adaboostm1', 'weka:attributeselectedclassifier', 'weka:bagging', 'weka:ibk', 'weka:j48', 'weka:multilayerperceptron', 'weka:randomforest']


In [51]:
df = df[df["familia"].notna()].copy()       
cont = df.groupby(["task_id", "familia"]).size().unstack(fill_value=0)
cob = (cont >= 3).mean()
fams_finais = cob[cob >= 0.70].index.tolist()

df.to_parquet("../data/runs_v3_familia.parquet")

resumo = (df.groupby("familia")
            .agg(biblioteca=("biblioteca", "first"), n_runs=("run_id", "size"),
                 n_bases_min1=("task_id", "nunique"))
            .assign(bases_min3=(cont >= 3).sum(),
                    cobertura_min3=cob,
                    final=lambda x: x.index.isin(fams_finais)))
resumo.to_csv("../data/familias.csv")
print(resumo["final"].sum(), "famílias finais")

22 famílias finais


In [52]:
fk = df[df["familia"] == "sklearn:FKCEigenPro"]

print("runs:", len(fk))
print("bases:", fk["task_id"].nunique())
print("flows distintos:", fk["flow_id"].nunique())
print("setups distintos:", fk["setup_id"].nunique())
print()
print(fk["flow_name"].iloc[0])
print()
print(fk.groupby("flow_id")["flow_name"].first().to_string())

runs: 50604
bases: 60
flows distintos: 3
setups distintos: 14913

sklearn.pipeline.Pipeline(simpleimputer=sklearn.impute._base.SimpleImputer,fkc_eigenpro=sklearn_extra.fast_kernel.FKC_EigenPro)(1)

flow_id
13291                                                             sklearn.pipeline.Pipeline(simpleimputer=sklearn.impute._base.SimpleImputer,fkc_eigenpro=sklearn_extra.fast_kernel.FKC_EigenPro)(1)
13293    sklearn.pipeline.Pipeline(simpleimputer=sklearn.impute._base.SimpleImputer,standardscaler=sklearn.preprocessing.data.StandardScaler,fkc_eigenpro=sklearn_extra.fast_kernel.FKC_EigenPro)(1)
15083      sklearn.pipeline.Pipeline(simpleimputer=sklearn.impute._base.SimpleImputer,standardscaler=sklearn.preprocessing.data.StandardScaler,fkceigenpro=sklearn_extra.fast_kernel.FKCEigenPro)(1)


## 4.Agregação e tabelas

In [53]:
df22 = df[df["familia"].isin(fams_finais)]
g = df22.groupby(["task_id", "familia"])["value"]
cel = pd.DataFrame({"n": g.size(), "max": g.max(), "mediana": g.median()}).reset_index()
cel = cel[cel["n"] >= 3]

from scipy.stats import spearmanr
print("células:", len(cel))
print("corr(n, max) geral:", spearmanr(cel["n"], cel["max"])[0])

#dentro de cada base: o n explica a ordem das famílias?
def corr_base(x):
    return spearmanr(x["n"], x["max"])[0]
por_base = cel.groupby("task_id").apply(corr_base)
print(por_base.describe())

#controle: a mesma correlação usando a mediana (que não deveria crescer com n)
print("corr(n, mediana) geral:", spearmanr(cel["n"], cel["mediana"])[0])

células: 1114
corr(n, max) geral: 0.17323213224451472
count    60.000000
mean      0.463769
std       0.183177
min       0.000000
25%       0.365614
50%       0.458840
75%       0.563366
max       0.904762
dtype: float64
corr(n, mediana) geral: 0.05063358855576164


In [54]:
def corr_base_med(x):
    return spearmanr(x["n"], x["mediana"])[0]
por_base_med = cel.groupby("task_id").apply(corr_base_med)
print(por_base_med.describe())

count    60.000000
mean      0.057928
std       0.255503
min      -0.650000
25%      -0.090016
50%       0.065830
75%       0.187620
max       0.821429
dtype: float64


In [55]:
print(por_base.sort_values().head(3))
print(cel[cel["task_id"] == por_base.sort_values().index[0]].shape)

task_id
14952     0.000000
167141    0.071429
31        0.165443
dtype: float64
(9, 5)


In [56]:
rng = np.random.default_rng(42)
R = 100

#Gera o max_sub3 das tabelas principais
def max_sub(valores, k=3, R=R):
    v = valores.to_numpy()
    return np.mean([rng.choice(v, size=k, replace=False).max() for _ in range(R)])

sub = (df22.groupby(["task_id", "familia"])["value"]
         .apply(lambda s: max_sub(s) if len(s) >= 3 else np.nan)
         .rename("max_sub3").reset_index())
cel = cel.merge(sub, on=["task_id", "familia"])

#quanto o ranking das famílias por base muda entre max completo e max_sub3
def corr_rank(x):
    return spearmanr(x["max"], x["max_sub3"])[0]
print(cel.groupby("task_id").apply(corr_rank).describe())

count    60.000000
mean      0.720186
std       0.126051
min       0.356296
25%       0.621927
50%       0.739620
75%       0.812253
max       0.942857
dtype: float64


In [57]:
#Só diagnóstico: robustez do k (compara max_sub3 com max_sub10)
def max_subk(valores, k, R=100):
    v = valores.to_numpy()
    return np.mean([rng.choice(v, size=k, replace=False).max() for _ in range(R)])

sub10 = (df22.groupby(["task_id", "familia"])["value"]
           .apply(lambda s: max_subk(s, 10) if len(s) >= 10 else np.nan)
           .rename("max_sub10").reset_index())
c10 = cel.merge(sub10, on=["task_id", "familia"]).dropna(subset=["max_sub10"])
print("células com n>=10:", len(c10), "de", len(cel))
print(c10.groupby("task_id").apply(lambda x: spearmanr(x["max_sub3"], x["max_sub10"])[0]
                                   if len(x) > 3 else np.nan).describe())

células com n>=10: 823 de 1114
count    60.000000
mean      0.902892
std       0.070512
min       0.674725
25%       0.860242
50%       0.924647
75%       0.953431
max       1.000000
dtype: float64


In [58]:
bruto = pd.read_parquet("../data/runs_consolidado.parquet")  # 3.624.525 runs, sem limpeza
#precisa da coluna familia: traga do df pelo flow_id
mapa = df[["flow_id", "familia"]].drop_duplicates("flow_id")
bruto = bruto.merge(mapa, on="flow_id", how="inner")
bruto = bruto[bruto["familia"].isin(fams_finais)]

g2 = bruto.groupby(["task_id", "familia"])["value"].agg(["size", "max"]).reset_index()
g2 = g2[g2["size"] >= 3].rename(columns={"max": "max_bruto"})
comp = cel[["task_id", "familia", "max"]].merge(g2[["task_id", "familia", "max_bruto"]],
                                                on=["task_id", "familia"])
comp["dif"] = comp["max_bruto"] - comp["max"]
print(comp["dif"].describe())
print(comp.groupby("task_id").apply(lambda x: spearmanr(x["max"], x["max_bruto"])[0]).describe())

count    1114.000000
mean        0.000001
std         0.000050
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.001671
Name: dif, dtype: float64
count    60.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [59]:
print(comp[comp["dif"] > 0].sort_values("dif", ascending=False))

     task_id            familia       max  max_bruto       dif
566     3560  weka:randomforest  0.518571   0.520242  0.001671


In [60]:
b = cel[cel["task_id"] == 3560].sort_values("max", ascending=False)
print(b[["familia", "n", "max", "mediana"]].to_string(index=False))

                               familia    n      max  mediana
sklearn:HistGradientBoostingClassifier  454 0.598969 0.550118
            sklearn:AdaBoostClassifier  546 0.593472 0.552336
        sklearn:RandomForestClassifier 5465 0.590636 0.568613
                           sklearn:SVC 4570 0.586267 0.525350
                          weka:bagging   70 0.586028 0.536428
    sklearn:GradientBoostingClassifier 1851 0.583860 0.499095
                    sklearn:GaussianNB    3 0.582664 0.582664
            sklearn:LogisticRegression  280 0.581999 0.580716
                         weka:logistic    6 0.581952 0.579437
                              weka:ibk    9 0.581789 0.532370
             weka:multilayerperceptron   14 0.581229 0.571336
                       weka:adaboostm1   70 0.578266 0.518881
                       weka:naivebayes    4 0.578266 0.578266
      weka:attributeselectedclassifier   16 0.578037 0.537802
                        mlr:naiveBayes    4 0.577783 0.577783
        

In [61]:
por_base = cel.groupby("task_id").agg(
    melhor=("max", "max"),
    mediana_dos_max=("max", "median"),
    amplitude=("max", lambda s: s.max() - s.min()),
    n_familias=("familia", "size"),
)
print(por_base.sort_values("melhor").head(10))
print(por_base["melhor"].describe())

           melhor  mediana_dos_max  amplitude  n_familias
task_id                                                  
3560     0.598969         0.579747   0.080398          22
125920   0.664811         0.634631   0.081740          11
23       0.749221         0.718944   0.093543          22
10101    0.753519         0.732830   0.111640          22
3904     0.770867         0.714550   0.179633          21
9971     0.780998         0.732209   0.168118          21
31       0.805433         0.779147   0.106385          22
37       0.842881         0.831952   0.109045          22
3913     0.847269         0.824085   0.147506          22
3917     0.848003         0.791368   0.219804          21
count    60.000000
mean      0.939227
std       0.090485
min       0.598969
25%       0.927536
50%       0.988829
75%       0.999493
max       1.000000
Name: melhor, dtype: float64


In [62]:
sat = por_base[por_base["melhor"] > 0.99]
print(len(sat), "bases com melhor > 0.99")
print(sat["amplitude"].describe())
# quantas famílias ficam a menos de 0.001 do melhor, por base
cel["gap"] = cel.groupby("task_id")["max"].transform("max") - cel["max"]
print(cel.assign(quase=cel["gap"] < 0.001).groupby("task_id")["quase"].sum().describe())

28 bases com melhor > 0.99
count    28.000000
mean      0.089694
std       0.065997
min       0.022905
25%       0.050640
50%       0.072551
75%       0.099559
max       0.281814
Name: amplitude, dtype: float64
count    60.000000
mean      3.583333
std       3.792462
min       1.000000
25%       1.000000
50%       2.000000
75%       5.000000
max      15.000000
Name: quase, dtype: float64


In [63]:
bases = sorted(df22["task_id"].unique())
fams = sorted(fams_finais)

def pivot(col):
    return (cel.pivot(index="task_id", columns="familia", values=col)
               .reindex(index=bases, columns=fams))

tab_sub3 = pivot("max_sub3")   # principal
tab_max  = pivot("max")        # sensibilidade
tab_med  = pivot("mediana")
tab_n    = pivot("n")

print(tab_sub3.shape)
print("NaN:", tab_sub3.isna().sum().sum(), "de", tab_sub3.size,
      f"({tab_sub3.isna().mean().mean():.1%})")
print("NaN por família:\n", tab_sub3.isna().sum().sort_values(ascending=False).head(8))
print("NaN por base:\n", tab_sub3.isna().sum(axis=1).sort_values(ascending=False).head(8))

(60, 22)
NaN: 206 de 1320 (15.6%)
NaN por família:
 familia
sklearn:GaussianNB                  18
mlr:naiveBayes                      17
weka:lwl                            17
weka:bagging                        14
weka:attributeselectedclassifier    14
weka:adaboostm1                     14
weka:multilayerperceptron           14
weka:naivebayes                     14
dtype: int64
NaN por base:
 task_id
146821    15
167119    15
167140    15
14954     14
167141    14
146824    14
146800    14
146817    13
dtype: int64


In [64]:
import os
os.makedirs("../data/tabelas", exist_ok=True)

tabelas = {
    "tab_sub3": tab_sub3,   # principal (max_sub3)
    "tab_max":  tab_max,    # sensibilidade (máximo completo)
    "tab_med":  tab_med,    # mediana
    "tab_n":    tab_n,      # nº de runs por célula
}

for nome, t in tabelas.items():
    t.to_parquet(f"../data/tabelas/{nome}.parquet")
    t.to_csv(f"../data/tabelas/{nome}.csv")

#a tabela longa também, caso precise refazer pivôs depois
cel.to_parquet("../data/tabelas/celulas_longa.parquet", index=False)

## 5. Faltantes e núcleo

In [65]:
t = pd.read_parquet("../data/tabelas/tab_sub3.parquet")
print(t.shape, t.index.name, t.equals(tab_sub3))

(60, 22) task_id True


In [66]:
nan_fam = tab_sub3.isna().sum().sort_values(ascending=False)
nan_base = tab_sub3.isna().sum(axis=1).sort_values(ascending=False)

print("NaN por família:\n", nan_fam.to_string())
print("\nNaN por base (top 10):\n", nan_base.head(10).to_string())
print("\nBases com >= 11 NaN (metade das famílias):", (nan_base >= 11).sum())

#os NaN são "aleatórios" ou concentrados na biblioteca?
bib = df22.drop_duplicates("familia").set_index("familia")["biblioteca"]
print("\nNaN por biblioteca:\n", nan_fam.groupby(bib).sum())
print("Famílias por biblioteca:\n", bib.value_counts())

NaN por família:
 familia
sklearn:GaussianNB                        18
mlr:naiveBayes                            17
weka:lwl                                  17
weka:bagging                              14
weka:attributeselectedclassifier          14
weka:adaboostm1                           14
weka:multilayerperceptron                 14
weka:naivebayes                           14
weka:ibk                                  14
weka:reptree                              14
weka:j48                                  13
mlr:rpart                                 13
weka:logistic                             12
weka:randomforest                         10
sklearn:GradientBoostingClassifier         7
sklearn:LogisticRegression                 1
sklearn:AdaBoostClassifier                 0
sklearn:DecisionTreeClassifier             0
sklearn:FKCEigenPro                        0
sklearn:SVC                                0
sklearn:HistGradientBoostingClassifier     0
sklearn:RandomForestClassifie

In [67]:
obs = tab_sub3.notna().astype(int)
comum = obs.values @ obs.values.T          #nº de famílias observadas em ambas
np.fill_diagonal(comum, 999)
print("mínimo de famílias em comum entre duas bases:", comum.min())
print("pares com < 5 em comum:", int(((comum < 5).sum()) / 2))

mínimo de famílias em comum entre duas bases: 6
pares com < 5 em comum: 0


In [68]:
fw = [f for f in tab_sub3.columns if f.startswith("weka:")]
fm = [f for f in tab_sub3.columns if f.startswith("mlr:")]
fs = [f for f in tab_sub3.columns if f.startswith("sklearn:")]

pb = pd.DataFrame({
    "nan_weka": tab_sub3[fw].isna().sum(axis=1),
    "nan_mlr": tab_sub3[fm].isna().sum(axis=1),
    "nan_sklearn": tab_sub3[fs].isna().sum(axis=1),
})
print(pb["nan_weka"].value_counts().sort_index())      #bimodal?
print(sorted(pb[pb["nan_weka"] >= 8].index))           #quais bases
print(pb.sort_values("nan_weka", ascending=False).head(15))

nan_weka
0     43
1      3
8      1
9      1
10     2
11    10
Name: count, dtype: int64
[14952, 14954, 125920, 125922, 146800, 146817, 146819, 146820, 146821, 146822, 146824, 167119, 167140, 167141]
         nan_weka  nan_mlr  nan_sklearn
task_id                                
146822         11        2            0
146821         11        2            2
146800         11        2            1
146817         11        2            0
167140         11        2            2
167141         11        2            1
167119         11        2            2
146824         11        2            1
146820         11        2            0
146819         11        2            0
14954          10        2            2
14952          10        1            2
125920          9        1            1
125922          8        1            1
9977            1        0            0


In [69]:
df22["upload_time"] = pd.to_datetime(df22["upload_time"])
grupo = (pb["nan_weka"] >= 8).map({True: "sem_weka", False: "com_weka"}).rename("grupo")

primeiro = df22.groupby("task_id")["upload_time"].min().rename("primeiro_run")
print(pd.concat([primeiro, grupo], axis=1).groupby("grupo")["primeiro_run"].describe())

print("weka, último run no df:", df22[df22.biblioteca == "weka"]["upload_time"].max())
print("mlr, último run no df:",  df22[df22.biblioteca == "mlr"]["upload_time"].max())

         count                        mean                  min  \
grupo                                                             
com_weka    46  2015-01-05 00:22:50.521739  2014-04-06 23:44:14   
sem_weka    14  2017-11-27 10:46:22.285714  2016-10-12 21:31:35   

                                 25%                         50%  \
grupo                                                              
com_weka  2014-04-07 01:03:12.250000  2014-10-08 13:49:42.500000   
sem_weka         2018-02-21 14:58:41  2018-02-21 14:59:53.500000   

                                 75%                  max  
grupo                                                      
com_weka  2015-12-01 21:33:32.500000  2016-04-28 17:06:30  
sem_weka  2018-02-21 17:01:43.250000  2018-04-11 23:50:51  
weka, último run no df: 2019-10-12 03:30:37
mlr, último run no df: 2022-03-15 09:25:16


In [70]:
core_b = pb.index[pb["nan_weka"] <= 1]
sub = tab_sub3.loc[core_b]
fam_ok = sub.columns[sub.isna().sum() == 0]
print(len(core_b), "bases;", len(fam_ok), "famílias sem NaN nelas")
print(sorted(fam_ok))
print("NaN restantes nas 46 bases:", int(sub.isna().sum().sum()))

46 bases; 17 famílias sem NaN nelas
['sklearn:AdaBoostClassifier', 'sklearn:DecisionTreeClassifier', 'sklearn:FKCEigenPro', 'sklearn:GradientBoostingClassifier', 'sklearn:HistGradientBoostingClassifier', 'sklearn:RandomForestClassifier', 'sklearn:SVC', 'weka:adaboostm1', 'weka:attributeselectedclassifier', 'weka:bagging', 'weka:ibk', 'weka:j48', 'weka:logistic', 'weka:multilayerperceptron', 'weka:naivebayes', 'weka:randomforest', 'weka:reptree']
NaN restantes nas 46 bases: 19


In [71]:
w = df22[df22.biblioteca == "weka"]
print(w.groupby(w["upload_time"].dt.year).size())

m = df22[df22.biblioteca == "mlr"]
print(m.groupby(m["upload_time"].dt.year).size())

# runs weka e mlr que caíram nas 14 bases do lote tardio
tardias = pb.index[pb["nan_weka"] >= 8]
print("runs weka nas 14 bases:", w["task_id"].isin(tardias).sum(), "de", len(w))
print("runs mlr nas 14 bases:", m["task_id"].isin(tardias).sum(), "de", len(m))

upload_time
2014    5671
2015    1434
2016    6614
2017    4849
2018      44
2019     375
dtype: int64
upload_time
2016     8157
2017    94702
2018       62
2019        4
2021       16
2022        1
dtype: int64
runs weka nas 14 bases: 104 de 18987
runs mlr nas 14 bases: 2057 de 102942


In [72]:
nucleo = tab_sub3.loc[core_b, fam_ok]
assert nucleo.isna().sum().sum() == 0
nucleo.to_parquet("../data/tabelas/nucleo_sub3.parquet")
nucleo.to_csv("../data/tabelas/nucleo_sub3.csv")
print(nucleo.shape)   # (46, 17)

(46, 17)


In [73]:
#Só diagnóstico: R = 200, semente por parâmetro. Usada na comparação com e sem limpeza e no ruído entre duas sementes.
def sub_mean_max(v, k=3, R=200, seed=0):
    if len(v) < k:
        return np.nan
    r = np.random.default_rng(seed)
    return np.mean([r.choice(v, size=k, replace=False).max() for _ in range(R)])

def tab_sub(frame, seed):
    return (frame.groupby(["task_id", "familia"])["value"]
                 .apply(lambda s: sub_mean_max(s.to_numpy(), seed=seed))
                 .rename(f"s{seed}"))

limpo = df22[df22["familia"].isin(fams_finais)]
a = tab_sub(limpo, 1)
b = tab_sub(limpo, 2)
ruido = (a - b).dropna()
print("ruído entre duas sementes (mesmos dados):")
print(ruido.abs().describe())

ruído entre duas sementes (mesmos dados):
count    1114.000000
mean        0.001144
std         0.002220
min         0.000000
25%         0.000037
50%         0.000331
75%         0.001393
max         0.028212
dtype: float64


## 6.Diagnósticos

In [74]:
# 'bruto' é o runs_consolidado já com a coluna familia, filtrado às 22 famílias (como no passo anterior)
chaves = cel[["task_id", "familia"]]            # as 1.114 células com n >= 3
c = tab_sub(limpo, 1).rename("limpo")
r_ = tab_sub(bruto, 1).rename("bruto")
comp = pd.concat([c, r_], axis=1).reset_index().merge(chaves, on=["task_id", "familia"]).dropna()
comp["dif"] = comp["limpo"] - comp["bruto"]

print(comp["dif"].describe())
print("diferença absoluta > ruído p95:", (comp["dif"].abs() > ruido.abs().quantile(0.95)).sum(), "células")
print(comp.groupby("task_id").apply(lambda x: spearmanr(x["limpo"], x["bruto"])[0]).describe())

count    1114.000000
mean        0.008637
std         0.031169
min        -0.004547
25%         0.000000
50%         0.000000
75%         0.000156
max         0.310023
Name: dif, dtype: float64
diferença absoluta > ruído p95: 173 células
count    60.000000
mean      0.954753
std       0.054700
min       0.690909
25%       0.940994
50%       0.976190
75%       0.989949
max       1.000000
dtype: float64


In [75]:
med_l = limpo.groupby(["task_id", "familia"])["value"].median().rename("med_limpo")
med_b = bruto.groupby(["task_id", "familia"])["value"].median().rename("med_bruto")
m = pd.concat([med_l, med_b], axis=1).reset_index().merge(chaves, on=["task_id", "familia"]).dropna()
m["dif"] = m["med_limpo"] - m["med_bruto"]
print(m["dif"].describe())
print(m.groupby("task_id").apply(lambda x: spearmanr(x["med_limpo"], x["med_bruto"])[0]).describe())

count    1114.000000
mean        0.019673
std         0.074884
min        -0.001450
25%         0.000000
50%         0.000000
75%         0.000076
max         0.490569
Name: dif, dtype: float64
count    60.000000
mean      0.951179
std       0.063077
min       0.640881
25%       0.933909
50%       0.971118
75%       0.996612
max       1.000000
dtype: float64


In [76]:
print("p95 do ruído:", ruido.abs().quantile(0.95).round(4))

# onde estão as diferenças grandes?
comp = comp.merge(bib.rename("biblioteca"), left_on="familia", right_index=True)
comp["grande"] = comp["dif"].abs() > ruido.abs().quantile(0.95)
print(comp.groupby("biblioteca")["grande"].agg(["sum", "size", "mean"]))
print(comp.sort_values("dif", ascending=False).head(10)[["task_id", "familia", "limpo", "bruto", "dif"]])

# as bases onde o ranking mais muda
rk = comp.groupby("task_id").apply(lambda x: spearmanr(x["limpo"], x["bruto"])[0])
print(rk.sort_values().head(5))

p95 do ruído: 0.0048
            sum  size      mean
biblioteca                     
mlr          22    90  0.244444
sklearn     135   514  0.262646
weka         16   510  0.031373
      task_id                             familia     limpo     bruto  \
569      3902                           mlr:rpart  0.886240  0.576217   
971     14965                           mlr:rpart  0.803915  0.537769   
374        49                           mlr:rpart  0.908665  0.688943   
590      3903                           mlr:rpart  0.711548  0.509845   
1025   125922  sklearn:GradientBoostingClassifier  0.956583  0.763204   
71         12  sklearn:GradientBoostingClassifier  0.966846  0.776754   
654      3917                           mlr:rpart  0.708134  0.519254   
933     10101                           mlr:rpart  0.708660  0.520691   
738      9952                           mlr:rpart  0.791013  0.606354   
1           3                           mlr:rpart  0.943861  0.769212   

           dif 

In [77]:
# 1) os runs removidos estão todos perto de 0,5? (a regra só deveria pegar 0,47 a 0,53)
removidos = bruto[~bruto["run_id"].isin(limpo["run_id"])]
print(removidos["value"].describe())
print("removidos acima de 0.53:", (removidos["value"] > 0.53).sum())

# 2) a diferença no max_sub3 acompanha a fração removida da célula?
nb = bruto.groupby(["task_id", "familia"]).size().rename("n_bruto")
nl = limpo.groupby(["task_id", "familia"]).size().rename("n_limpo")
fr = pd.concat([nb, nl], axis=1).reset_index()
fr["frac_removida"] = 1 - fr["n_limpo"] / fr["n_bruto"]
x = comp.merge(fr, on=["task_id", "familia"])
print(spearmanr(x["frac_removida"], x["dif"]))
print(x.groupby(pd.cut(x["frac_removida"], [-0.01, 0, 0.1, 0.3, 0.6, 1]))["dif"].agg(["size", "mean", "max"]))

tab_sub3_bruto = comp.pivot(index="task_id", columns="familia", values="bruto").reindex(index=bases, columns=fams)
tab_sub3_bruto.to_parquet("../data/tabelas/tab_sub3_sem_limpeza.parquet")

count    501597.000000
mean          0.497058
std           0.004281
min           0.485371
25%           0.495506
50%           0.499034
75%           0.500000
max           0.526733
Name: value, dtype: float64
removidos acima de 0.53: 0
SignificanceResult(statistic=np.float64(0.7604955150232281), pvalue=np.float64(9.307144645073757e-211))
               size      mean       max
frac_removida                          
(-0.01, 0.0]    747  0.000000  0.000000
(0.0, 0.1]      189  0.001358  0.028823
(0.1, 0.3]       84  0.012039  0.040845
(0.3, 0.6]       63  0.055356  0.171663
(0.6, 1.0]       31  0.156988  0.310023


In [78]:
print("removidos > 0.51:", (removidos["value"] > 0.51).sum(), "de", len(removidos))
print("removidos > 0.52:", (removidos["value"] > 0.52).sum())

alto = removidos[removidos["value"] > 0.52]
print(alto.groupby(["task_id", "familia"]).size().sort_values(ascending=False).head(10))

c = comp[comp["task_id"].isin([219, 125920, 3560]) & (comp["dif"].abs() > 0.0048)]
print(c[["task_id", "familia", "limpo", "bruto", "dif"]].sort_values("dif", ascending=False))

removidos > 0.51: 2364 de 501597
removidos > 0.52: 542
task_id  familia                           
125920   sklearn:DecisionTreeClassifier        188
3560     sklearn:DecisionTreeClassifier        119
         sklearn:SVC                           100
219      sklearn:RandomForestClassifier         72
125920   sklearn:SVC                            23
3560     weka:adaboostm1                         8
         sklearn:AdaBoostClassifier              6
219      sklearn:AdaBoostClassifier              5
3560     sklearn:GradientBoostingClassifier      4
125920   sklearn:GradientBoostingClassifier      3
dtype: int64
      task_id                             familia     limpo     bruto  \
423       219  sklearn:GradientBoostingClassifier  0.880154  0.758107   
1014   125920  sklearn:GradientBoostingClassifier  0.596952  0.549160   
418       219                           mlr:rpart  0.736697  0.698139   
1018   125920                         sklearn:SVC  0.590539  0.575266   
420       219

In [79]:
cel["r_max"] = cel.groupby("task_id")["max"].rank(ascending=False, method="average")
cel["r_sub3"] = cel.groupby("task_id")["max_sub3"].rank(ascending=False, method="average")
cel["mov"] = cel["r_max"] - cel["r_sub3"]      # positivo: sobe com a subamostra

# por família: quem sobe e quem desce
resumo = (cel.groupby("familia")
             .agg(mov_medio=("mov", "mean"), n_runs_med=("n", "median"), bases=("task_id", "size"))
             .sort_values("mov_medio"))
print(resumo.to_string())

fk = cel[cel["familia"] == "sklearn:FKCEigenPro"]
print("\nFKCEigenPro: posto médio com max =", fk["r_max"].mean().round(2),
      "| com max_sub3 =", fk["r_sub3"].mean().round(2))
print(fk["mov"].describe())

                                        mov_medio  n_runs_med  bases
familia                                                             
sklearn:SVC                            -10.141667      5985.0     60
sklearn:GradientBoostingClassifier      -6.716981       999.0     53
weka:attributeselectedclassifier        -3.347826        49.5     46
weka:adaboostm1                         -0.989130        78.5     46
weka:bagging                            -0.978261        74.5     46
sklearn:AdaBoostClassifier              -0.733333       422.5     60
mlr:rpart                               -0.574468       305.0     47
sklearn:RandomForestClassifier           0.100000      3483.0     60
sklearn:DecisionTreeClassifier           0.266667      1474.0     60
sklearn:HistGradientBoostingClassifier   0.508333       488.0     60
weka:j48                                 0.723404        21.0     47
sklearn:FKCEigenPro                      0.733333       497.5     60
weka:lwl                          

In [80]:
# 1) o melhor classificador de cada base muda?
top_max = cel.loc[cel.groupby("task_id")["max"].idxmax()].set_index("task_id")["familia"]
top_sub = cel.loc[cel.groupby("task_id")["max_sub3"].idxmax()].set_index("task_id")["familia"]
print("bases em que o melhor (top-1) é o mesmo:", (top_max == top_sub).mean().round(3))
print("\ntop-1 com max:\n", top_max.value_counts().head(6))
print("\ntop-1 com max_sub3:\n", top_sub.value_counts().head(6))

# 2) posto médio de cada família nas duas métricas
pm = cel.groupby("familia")[["r_max", "r_sub3"]].mean().round(2).sort_values("r_max")
print(pm.to_string())

bases em que o melhor (top-1) é o mesmo: 0.25

top-1 com max:
 familia
sklearn:HistGradientBoostingClassifier    20
sklearn:SVC                               20
sklearn:RandomForestClassifier             8
sklearn:AdaBoostClassifier                 7
sklearn:GradientBoostingClassifier         1
weka:attributeselectedclassifier           1
Name: count, dtype: int64

top-1 com max_sub3:
 familia
sklearn:HistGradientBoostingClassifier    22
weka:randomforest                         12
sklearn:RandomForestClassifier             9
sklearn:LogisticRegression                 8
weka:multilayerperceptron                  4
weka:logistic                              2
Name: count, dtype: int64
                                        r_max  r_sub3
familia                                              
sklearn:HistGradientBoostingClassifier   3.34    2.83
sklearn:RandomForestClassifier           3.53    3.43
sklearn:SVC                              4.54   14.68
weka:bagging                         

In [81]:
def regret(avaliar_em, escolher_por):
    out = []
    for t, g in cel.groupby("task_id"):
        pick = g.loc[g[escolher_por].idxmax()]
        out.append(g[avaliar_em].max() - pick[avaliar_em])
    return pd.Series(out)

print("escolher por max_sub3, avaliar em max:\n", regret("max", "max_sub3").describe())
print("escolher por max, avaliar em max_sub3:\n", regret("max_sub3", "max").describe())

def top3(col):
    return (cel.sort_values(["task_id", col], ascending=[True, False])
               .groupby("task_id").head(3).groupby("task_id")["familia"].apply(set))
a, b = top3("max"), top3("max_sub3")
print(pd.Series([len(a[i] & b[i]) for i in a.index]).value_counts().sort_index())

escolher por max_sub3, avaliar em max:
 count    60.000000
mean      0.003020
std       0.004974
min       0.000000
25%       0.000000
50%       0.000731
75%       0.004436
max       0.027587
dtype: float64
escolher por max, avaliar em max_sub3:
 count    60.000000
mean      0.016309
std       0.024599
min       0.000000
25%       0.000106
50%       0.007382
75%       0.020085
max       0.127386
dtype: float64


0     7
1    19
2    23
3    11
Name: count, dtype: int64


In [82]:
melhor = tab_max.max(axis=1)
fracas = melhor[melhor < 0.70].index.tolist()
print("bases fracas:", fracas)          # esperado: [3560, 125920]

tab_sub3_sem_fracas = tab_sub3.drop(index=fracas)
nucleo_sem_fracas = nucleo.drop(index=[b for b in fracas if b in nucleo.index])
print(tab_sub3_sem_fracas.shape, nucleo_sem_fracas.shape)   # esperado: (58, 22) (45, 17)

tab_sub3_sem_fracas.to_parquet("../data/tabelas/tab_sub3_sem_fracas.parquet")
nucleo_sem_fracas.to_parquet("../data/tabelas/nucleo_sub3_sem_fracas.parquet")

bases fracas: [3560, 125920]
(58, 22) (45, 17)


In [83]:
import os
print(sorted(os.listdir("../data/tabelas")))

['celulas_longa.parquet', 'nucleo_sub3.csv', 'nucleo_sub3.parquet', 'nucleo_sub3_sem_fracas.parquet', 'tab_max.csv', 'tab_max.parquet', 'tab_med.csv', 'tab_med.parquet', 'tab_n.csv', 'tab_n.parquet', 'tab_sub3.csv', 'tab_sub3.parquet', 'tab_sub3_sem_fracas.parquet', 'tab_sub3_sem_limpeza.parquet']


In [84]:
import pandas as pd, numpy as np, openml
from scipy.stats import mannwhitneyu

tab = pd.read_parquet("../data/tabelas/tab_sub3.parquet")   # ajuste o formato/caminho
tab_n = pd.read_parquet("../data/tabelas/tab_n.parquet")

#1) grupos: bases sem nenhuma família weka vs com weka
weka_cols = [c for c in tab.columns if c.startswith("weka:")]
sem_weka = tab[weka_cols].isna().all(axis=1)
print(sem_weka.value_counts())          # esperado: 14 vs 46

#2) metadados das bases (instâncias, features)
tasks = openml.tasks.list_tasks(task_id=list(tab.index), output_format="dataframe")
ds = openml.datasets.list_datasets(
        data_id=list(tasks["did"]), output_format="dataframe")
meta = (tasks[["tid","did"]].merge(ds[["did","NumberOfInstances","NumberOfFeatures",
                                       "NumberOfClasses","NumberOfSymbolicFeatures"]],
                                   on="did").set_index("tid"))

#3) descritores de desempenho e volume (só famílias sklearn, comuns aos dois grupos)
sk_cols = [c for c in tab.columns if c.startswith("sklearn:")]
res = meta.reindex(tab.index).copy()
res["grupo"] = np.where(sem_weka, "sem_weka", "com_weka")
res["melhor_max"] = tab[sk_cols].max(axis=1)
res["mediana_max_sk"] = tab[sk_cols].median(axis=1)
res["amplitude_sk"] = tab[sk_cols].max(axis=1) - tab[sk_cols].min(axis=1)
res["runs_total"] = tab_n.sum(axis=1)
res["frac_categ"] = res["NumberOfSymbolicFeatures"] / res["NumberOfFeatures"]
res["task_id"] = res.index

vars_ = ["NumberOfInstances","NumberOfFeatures","NumberOfClasses","frac_categ",
         "melhor_max","mediana_max_sk","amplitude_sk","runs_total","task_id"]

#4) comparação: mediana por grupo + Mann-Whitney + efeito (probabilidade de superioridade)
linhas = []
for v in vars_:
    a = res.loc[res.grupo=="sem_weka", v].dropna()
    b = res.loc[res.grupo=="com_weka", v].dropna()
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    linhas.append({"variavel": v, "med_sem_weka": a.median(), "med_com_weka": b.median(),
                   "p": round(p,3), "prob_sup": round(u/(len(a)*len(b)),2)})
print(pd.DataFrame(linhas).to_string(index=False))

False    50
True     10
Name: count, dtype: int64
         variavel  med_sem_weka  med_com_weka     p  prob_sup
NumberOfInstances   2155.000000   2000.000000 0.655      0.55
 NumberOfFeatures     21.000000     22.000000 1.000      0.50
  NumberOfClasses      3.500000      2.000000 0.357      0.59
       frac_categ      0.101916      0.079412 0.655      0.55
       melhor_max      0.991032      0.966889 0.162      0.64
   mediana_max_sk      0.951289      0.931888 0.558      0.56
     amplitude_sk      0.160917      0.103366 0.271      0.61
       runs_total   7959.500000  17374.000000 0.000      0.03
          task_id 146821.500000   3554.500000 0.000      1.00


In [85]:
tab = pd.read_parquet("../data/tabelas/tab_sub3.parquet")       # 60 x 22
tab_n = pd.read_parquet("../data/tabelas/tab_n.parquet")
nucleo = pd.read_parquet("../data/tabelas/nucleo_sub3.parquet") # 46 x 17 (ajuste o nome)

#1) grupos: dentro do núcleo vs fora do núcleo
fora = ~tab.index.isin(nucleo.index)
print(pd.Series(np.where(fora, "fora_nucleo", "nucleo")).value_counts())

#2) metadados das bases
tasks = openml.tasks.list_tasks(task_id=list(tab.index), output_format="dataframe")
ds = openml.datasets.list_datasets(
        data_id=list(tasks["did"]), output_format="dataframe")
meta = (tasks[["tid","did"]]
        .merge(ds[["did","NumberOfInstances","NumberOfFeatures",
                   "NumberOfClasses","NumberOfSymbolicFeatures"]], on="did")
        .set_index("tid"))

#3) descritores (só sklearn: observado nos dois grupos)
sk_cols = [c for c in tab.columns if c.startswith("sklearn:")]
res = meta.reindex(tab.index).copy()
res["grupo"] = np.where(fora, "fora_nucleo", "nucleo")
res["melhor_max"] = tab[sk_cols].max(axis=1)
res["mediana_max_sk"] = tab[sk_cols].median(axis=1)
res["amplitude_sk"] = tab[sk_cols].max(axis=1) - tab[sk_cols].min(axis=1)
res["runs_total"] = tab_n.sum(axis=1)
res["frac_categ"] = res["NumberOfSymbolicFeatures"] / res["NumberOfFeatures"]
res["task_id"] = res.index

vars_ = ["NumberOfInstances","NumberOfFeatures","NumberOfClasses","frac_categ",
         "melhor_max","mediana_max_sk","amplitude_sk","runs_total","task_id"]

#4) comparação
linhas = []
for v in vars_:
    a = res.loc[res.grupo=="fora_nucleo", v].dropna()
    b = res.loc[res.grupo=="nucleo", v].dropna()
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    linhas.append({"variavel": v,
                   "med_fora": a.median(), "med_nucleo": b.median(),
                   "p": round(p, 3), "prob_sup": round(u/(len(a)*len(b)), 2)})
print(pd.DataFrame(linhas).to_string(index=False))

#5) quais bases ficaram fora para registrar

nucleo         46
fora_nucleo    14
Name: count, dtype: int64
         variavel      med_fora   med_nucleo     p  prob_sup
NumberOfInstances   2155.000000  2000.000000 0.916      0.51
 NumberOfFeatures     24.500000    22.000000 0.760      0.53
  NumberOfClasses      3.000000     2.000000 0.604      0.54
       frac_categ      0.154762     0.058824 0.224      0.61
       melhor_max      0.991032     0.966889 0.238      0.61
   mediana_max_sk      0.951289     0.931888 0.606      0.55
     amplitude_sk      0.141347     0.107478 0.951      0.51
       runs_total   8133.000000 17937.000000 0.000      0.05
          task_id 146819.500000  3021.500000 0.000      0.99


In [86]:
   print(res.loc[fora & (res.index < 100000),
                 ["task_id","NumberOfInstances","NumberOfFeatures","NumberOfClasses","melhor_max"]])

         task_id  NumberOfInstances  NumberOfFeatures  NumberOfClasses  \
task_id                                                                  
14952      14952            11055.0              31.0              2.0   
14954      14954              540.0              40.0              2.0   

         melhor_max  
task_id              
14952      0.996867  
14954      0.915346  


In [87]:
for t in [14952, 14954]:
    print(t, tab.columns[tab.loc[t].isna()].tolist())

14952 ['mlr:naiveBayes', 'sklearn:GaussianNB', 'sklearn:GradientBoostingClassifier', 'weka:adaboostm1', 'weka:attributeselectedclassifier', 'weka:bagging', 'weka:ibk', 'weka:j48', 'weka:logistic', 'weka:lwl', 'weka:multilayerperceptron', 'weka:naivebayes', 'weka:reptree']
14954 ['mlr:naiveBayes', 'mlr:rpart', 'sklearn:GaussianNB', 'sklearn:GradientBoostingClassifier', 'weka:adaboostm1', 'weka:attributeselectedclassifier', 'weka:bagging', 'weka:ibk', 'weka:j48', 'weka:logistic', 'weka:lwl', 'weka:multilayerperceptron', 'weka:naivebayes', 'weka:reptree']


In [88]:
cols = ['weka:j48', 'sklearn:GaussianNB', 'sklearn:GradientBoostingClassifier', 'mlr:naiveBayes']
print(tab_n.loc[[14952, 14954], cols])

familia  weka:j48  sklearn:GaussianNB  sklearn:GradientBoostingClassifier  \
task_id                                                                     
14952         NaN                 NaN                                 NaN   
14954         NaN                 NaN                                 NaN   

familia  mlr:naiveBayes  
task_id                  
14952               NaN  
14954               NaN  


In [89]:
cols = ['weka:j48', 'sklearn:GaussianNB',
        'sklearn:GradientBoostingClassifier', 'mlr:naiveBayes']
c = (df22[df22.task_id.isin([14952, 14954]) & df22.familia.isin(cols)]
       .groupby(["task_id", "familia"]).size())
print(c if len(c) else "nenhum run nessas células")

task_id  familia                           
14952    sklearn:GaussianNB                    2
         sklearn:GradientBoostingClassifier    2
         weka:j48                              2
14954    sklearn:GaussianNB                    1
         sklearn:GradientBoostingClassifier    1
         weka:j48                              2
dtype: int64


In [90]:
#Só diagnóstico: ruído entre 5 sementes, R = 100.
def max_sub_seed(valores, seed, k=3, R=100):
    rng_l = np.random.default_rng(seed)
    v = valores.to_numpy()
    return np.mean([rng_l.choice(v, size=k, replace=False).max() for _ in range(R)])

g = df22.groupby(["task_id", "familia"])["value"]
n = g.size()
elig = n[n >= 3].index

M = pd.DataFrame({
    seed: pd.Series({idx: max_sub_seed(g.get_group(idx), seed) for idx in elig})
    for seed in range(5)          # 5 sementes; sobe para 10 se rodar rápido
})

ruido = M.std(axis=1)             # desvio-padrão por célula entre sementes
print(ruido.describe(percentiles=[.5, .95]))
print("maior amplitude (máx - mín) entre sementes:", (M.max(axis=1) - M.min(axis=1)).max())

count    1114.000000
mean        0.001493
std         0.002319
min         0.000000
50%         0.000661
95%         0.005913
max         0.021670
dtype: float64
maior amplitude (máx - mín) entre sementes: 0.05847094825693189


In [91]:
tab = pd.read_parquet("../data/tabelas/tab_sub3.parquet")
gaps = tab.apply(lambda r: np.diff(np.sort(r.dropna().to_numpy())[::-1]), axis=1)
todos = np.concatenate(gaps.to_numpy())
print(pd.Series(np.abs(todos)).describe(percentiles=[.25, .5, .75]))

count    1.054000e+03
mean     8.366509e-03
std      1.610307e-02
min      7.194577e-07
25%      7.800339e-04
50%      2.988068e-03
75%      8.661645e-03
max      1.796935e-01
dtype: float64


In [92]:
from scipy.special import comb

def emax_exact(valores, k=3):
    v = np.sort(valores.to_numpy())
    n = len(v)
    i = np.arange(1, n + 1)
    return (v * comb(i - 1, k - 1) / comb(n, k)).sum()

In [93]:
from scipy.stats import spearmanr
r = []
for t, grp in M.groupby(level=0):
    if len(grp) > 3:
        r.append(spearmanr(grp[0], grp[1])[0])
print(pd.Series(r).describe())

count    60.000000
mean      0.989505
std       0.011349
min       0.936364
25%       0.985037
50%       0.992151
75%       0.996707
max       1.000000
dtype: float64


In [95]:
import openml
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

#1) Contagem exata de tasks binárias vs. multiclasse nas 60 selecionadas
print("=== 1. DISTRIBUIÇÃO DE CLASSES NAS 60 TASKS ===")
n_multi = (selected_tasks["NumberOfClasses"] > 2).sum()
n_bin = (selected_tasks["NumberOfClasses"] == 2).sum()
print(f"Binárias: {n_bin} | Multiclasse: {n_multi}")
print(selected_tasks[selected_tasks["NumberOfClasses"] > 2][["tid", "name", "NumberOfClasses"]].to_string(index=False))

#2) Diagnóstico do resíduo da task 3022 (8,96% < 0,5) e dos removidos > 0,52 na base 219
print("\n=== 2. RESÍDUO TASK 3022 E BASE 219 ===")
flows_map = pd.read_csv("../data/flows.csv").set_index("flow_id")["flow_name"]

res_3022 = d3[(d3["task_id"] == 3022) & (d3["value"] < 0.5)].copy()
res_3022["flow_name"] = res_3022["flow_id"].map(flows_map)
print("Top flows com AUC < 0,5 restantes na task 3022:")
print(res_3022["flow_name"].value_counts().head(8))

rem_219 = removidos[(removidos["task_id"] == 219) & (removidos["value"] > 0.52)].copy()
rem_219["flow_name"] = rem_219["flow_id"].map(flows_map)
print("\nFlows removidos com AUC > 0,52 na base 219:")
print(rem_219["flow_name"].value_counts().head(8))

#3) Teste da hipótese do preditor constante (prior de treino por fold) na task 3903
print("\n=== 3. HIPÓTESE DO PREDITOR CONSTANTE (TASK 3903) ===")
task_3903 = openml.tasks.get_task(3903)
X, y = task_3903.get_X_and_y(dataset_format="dataframe")
y_bin = (y == y.unique()[1]).astype(int).to_numpy()

y_true_all, y_prob_all = [], []
_, _, _ = task_3903.get_split_dimensions()
for fold in range(10):
    train_idx, test_idx = task_3903.get_train_test_split_indices(fold=fold)
    prior_treino = y_bin[train_idx].mean()
    y_true_all.extend(y_bin[test_idx])
    y_prob_all.extend([prior_treino] * len(test_idx))

auc_constante = roc_auc_score(y_true_all, y_prob_all)
print(f"AUC teórica da prior de treino por fold na task 3903: {auc_constante:.6f} (modal observada no OpenML: 0.4993)")

=== 1. DISTRIBUIÇÃO DE CLASSES NAS 60 TASKS ===
Binárias: 31 | Multiclasse: 29
   tid                                   name  NumberOfClasses
     6                                 letter               26
    11                          balance-scale                3
    12                          mfeat-factors               10
    14                          mfeat-fourier               10
    16                         mfeat-karhunen               10
    18                    mfeat-morphological               10
    22                          mfeat-zernike               10
    23                                    cmc                3
    28                              optdigits               10
    32                              pendigits               10
    45                                 splice                3
    53                                vehicle                4
  2074                               satimage                6
  2079                             euca